# ETL & Data Cleansing.

### Import libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

### Define project paths

In [3]:
RAW_DATA_PATH = Path("../data/raw")
CLEANED_DATA_PATH = Path("../data/cleaned")
STAGING_DATA_PATH = Path("../data/staging")

CLEANED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

STAGING_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Raw:", RAW_DATA_PATH.resolve())
print("Cleaned:", CLEANED_DATA_PATH.resolve())
print("Staging:", STAGING_DATA_PATH.resolve())

Raw: C:\New folder\Hands on Projects\healthcare-operations-population-health-analytics\data\raw
Cleaned: C:\New folder\Hands on Projects\healthcare-operations-population-health-analytics\data\cleaned
Staging: C:\New folder\Hands on Projects\healthcare-operations-population-health-analytics\data\staging


### Load the raw sources again

In [4]:
patients_raw = pd.read_csv(
    RAW_DATA_PATH / "patients.csv"
)

providers_raw = pd.read_csv(
    RAW_DATA_PATH / "providers.csv"
)

departments_raw = pd.read_csv(
    RAW_DATA_PATH / "departments.csv"
)

payers_raw = pd.read_csv(
    RAW_DATA_PATH / "payers.csv"
)

diagnoses_raw = pd.read_csv(
    RAW_DATA_PATH / "diagnoses.csv"
)

procedures_raw = pd.read_csv(
    RAW_DATA_PATH / "procedures.csv"
)

encounters_raw = pd.read_csv(
    RAW_DATA_PATH / "encounters.csv"
)

encounter_diagnoses_raw = pd.read_csv(
    RAW_DATA_PATH / "encounter_diagnoses.csv"
)

encounter_procedures_raw = pd.read_csv(
    RAW_DATA_PATH / "encounter_procedures.csv"
)

admissions_raw = pd.read_csv(
    RAW_DATA_PATH / "admissions.csv"
)

appointments_raw = pd.read_csv(
    RAW_DATA_PATH / "appointments.csv"
)

lab_results_raw = pd.read_csv(
    RAW_DATA_PATH / "lab_results.csv"
)

print("All raw datasets loaded.")

All raw datasets loaded.


### Create working copies

In [5]:
patients_clean = patients_raw.copy()
providers_clean = providers_raw.copy()
departments_clean = departments_raw.copy()
payers_clean = payers_raw.copy()
diagnoses_clean = diagnoses_raw.copy()
procedures_clean = procedures_raw.copy()
encounters_clean = encounters_raw.copy()
encounter_diagnoses_clean = encounter_diagnoses_raw.copy()
encounter_procedures_clean = encounter_procedures_raw.copy()
admissions_clean = admissions_raw.copy()
appointments_clean = appointments_raw.copy()
lab_results_clean = lab_results_raw.copy()

### Create a cleaning audit log

In [6]:
cleaning_log = []

def log_cleaning_action(
    step,
    table,
    field,
    action,
    affected_rows,
    reason
):
    cleaning_log.append({
        "Step": step,
        "Table": table,
        "Field": field,
        "Action": action,
        "AffectedRows": int(affected_rows),
        "Reason": reason
    })

### Define cleaning order

1. departments
2. payers
3. diagnoses
4. procedures
5. providers
6. patients

7. encounters

8. encounter_diagnoses
9. encounter_procedures

10. admissions
11. appointments
12. lab_results

### Record before-cleaning row counts

In [7]:
clean_datasets = {
    "patients": patients_clean,
    "providers": providers_clean,
    "departments": departments_clean,
    "payers": payers_clean,
    "diagnoses": diagnoses_clean,
    "procedures": procedures_clean,
    "encounters": encounters_clean,
    "encounter_diagnoses": encounter_diagnoses_clean,
    "encounter_procedures": encounter_procedures_clean,
    "admissions": admissions_clean,
    "appointments": appointments_clean,
    "lab_results": lab_results_clean
}

In [8]:
before_cleaning_counts = pd.DataFrame([
    {
        "Table": table_name,
        "RowsBeforeCleaning": len(df)
    }
    for table_name, df in clean_datasets.items()
])

before_cleaning_counts

,Table,RowsBeforeCleaning
0,patients,10000
1,providers,120
2,departments,25
3,payers,6
4,diagnoses,50
5,procedures,40
6,encounters,90000
7,encounter_diagnoses,153595
8,encounter_procedures,70419
9,admissions,17740


### Create a quarantine folder

In [9]:
QUARANTINE_PATH = Path(
    "../data/processed/quarantine"
)

QUARANTINE_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print(QUARANTINE_PATH.resolve())

C:\New folder\Hands on Projects\healthcare-operations-population-health-analytics\data\processed\quarantine


### Standardize whitespace in reference tables

In [10]:
def strip_text_columns(df, table_name):

    for column in df.select_dtypes(
        include="object"
    ).columns:

        before = df[column].copy()

        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
        )

        changed = (
            before.astype("string")
            != df[column]
        ).fillna(False).sum()

        if changed > 0:
            log_cleaning_action(
                "5.2",
                table_name,
                column,
                "Trimmed leading/trailing whitespace",
                changed,
                "Text standardization"
            )

    return df

In [11]:
departments_clean = strip_text_columns(
    departments_clean,
    "departments"
)

payers_clean = strip_text_columns(
    payers_clean,
    "payers"
)

diagnoses_clean = strip_text_columns(
    diagnoses_clean,
    "diagnoses"
)

procedures_clean = strip_text_columns(
    procedures_clean,
    "procedures"
)

providers_clean = strip_text_columns(
    providers_clean,
    "providers"
)

patients_clean = strip_text_columns(
    patients_clean,
    "patients"
)

C:\Users\Aakash\AppData\Local\Temp\ipykernel_22992\3798032650.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df.select_dtypes(
C:\Users\Aakash\AppData\Local\Temp\ipykernel_22992\3798032650.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.h

### Clean departments

In [12]:
departments_clean["State"] = (
    departments_clean["State"]
    .str.upper()
)

In [13]:
print(
    "Missing DepartmentID:",
    departments_clean[
        "DepartmentID"
    ].isna().sum()
)

print(
    "Duplicate DepartmentID:",
    departments_clean[
        "DepartmentID"
    ].duplicated().sum()
)

Missing DepartmentID: 0
Duplicate DepartmentID: 0


In [14]:
valid_department_types = [
    "Emergency",
    "Inpatient",
    "Specialty",
    "Primary Care",
    "Outpatient",
    "Behavioral Health"
]

print(
    departments_clean[
        "DepartmentType"
    ].value_counts()
)

print(
    "Invalid DepartmentType:",
    (
        ~departments_clean[
            "DepartmentType"
        ].isin(valid_department_types)
    ).sum()
)

DepartmentType
Specialty            10
Outpatient            5
Emergency             3
Inpatient             3
Primary Care          2
Behavioral Health     2
Name: count, dtype: Int64
Invalid DepartmentType: 0


### Clean payers

In [15]:
print(
    "Duplicate PayerID:",
    payers_clean[
        "PayerID"
    ].duplicated().sum()
)

print(
    payers_clean[
        "PayerType"
    ].value_counts()
)

Duplicate PayerID: 0
PayerType
Government    2
Commercial    2
Self Pay      1
Other         1
Name: count, dtype: Int64


In [16]:
valid_payer_types = [
    "Government",
    "Commercial",
    "Self Pay",
    "Other"
]

print(
    "Invalid PayerType:",
    (
        ~payers_clean[
            "PayerType"
        ].isin(valid_payer_types)
    ).sum()
)

Invalid PayerType: 0


### Clean diagnoses

In [17]:
diagnoses_clean["DiagnosisCode"] = (
    diagnoses_clean[
        "DiagnosisCode"
    ]
    .str.upper()
)

In [18]:
diagnoses_clean[
    "ChronicConditionFlag"
] = pd.to_numeric(
    diagnoses_clean[
        "ChronicConditionFlag"
    ],
    errors="coerce"
).astype("Int64")

In [19]:
print(
    "Duplicate DiagnosisID:",
    diagnoses_clean[
        "DiagnosisID"
    ].duplicated().sum()
)

print(
    "Invalid ChronicConditionFlag:",
    (
        ~diagnoses_clean[
            "ChronicConditionFlag"
        ].isin([0, 1])
    ).sum()
)

Duplicate DiagnosisID: 0
Invalid ChronicConditionFlag: 0


### Clean procedures

In [20]:
procedures_clean[
    "ProcedureCode"
] = (
    procedures_clean[
        "ProcedureCode"
    ]
    .str.upper()
)

In [21]:
procedures_clean[
    "StandardCost"
] = pd.to_numeric(
    procedures_clean[
        "StandardCost"
    ],
    errors="coerce"
)

In [22]:
print(
    "Duplicate ProcedureID:",
    procedures_clean[
        "ProcedureID"
    ].duplicated().sum()
)

print(
    "Missing StandardCost:",
    procedures_clean[
        "StandardCost"
    ].isna().sum()
)

print(
    "Negative StandardCost:",
    (
        procedures_clean[
            "StandardCost"
        ] < 0
    ).sum()
)

Duplicate ProcedureID: 0
Missing StandardCost: 0
Negative StandardCost: 0


### Clean providers

In [23]:
providers_clean[
    "HireDate"
] = pd.to_datetime(
    providers_clean[
        "HireDate"
    ],
    errors="coerce"
)

In [24]:
providers_clean[
    "ActiveFlag"
] = pd.to_numeric(
    providers_clean[
        "ActiveFlag"
    ],
    errors="coerce"
).astype("Int64")

In [25]:
provider_type_map = {
    "physician": "Physician",
    "nurse practitioner": "Nurse Practitioner",
    "physician assistant": "Physician Assistant"
}

providers_clean[
    "ProviderType"
] = (
    providers_clean[
        "ProviderType"
    ]
    .str.lower()
    .map(provider_type_map)
    .fillna(
        providers_clean[
            "ProviderType"
        ]
    )
)

In [26]:
invalid_provider_departments = (
    ~providers_clean[
        "DepartmentID"
    ].isin(
        departments_clean[
            "DepartmentID"
        ]
    )
).sum()

print(
    "Invalid provider DepartmentID:",
    invalid_provider_departments
)

Invalid provider DepartmentID: 0


In [27]:
print(
    "Invalid ActiveFlag:",
    (
        ~providers_clean[
            "ActiveFlag"
        ].isin([0, 1])
    ).sum()
)

Invalid ActiveFlag: 0


### Convert patient dates

In [28]:
patients_clean[
    "BirthDate"
] = pd.to_datetime(
    patients_clean[
        "BirthDate"
    ],
    errors="coerce"
)

patients_clean[
    "RegistrationDate"
] = pd.to_datetime(
    patients_clean[
        "RegistrationDate"
    ],
    errors="coerce"
)

C:\Users\Aakash\AppData\Local\Temp\ipykernel_22992\3009446666.py:3: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  ] = pd.to_datetime(


In [29]:
log_cleaning_action(
    "5.2",
    "patients",
    "BirthDate / RegistrationDate",
    "Converted text dates to datetime",
    len(patients_clean),
    "Required for temporal validation and age calculations"
)

### Standardize Gender

In [30]:
gender_map = {
    "M": "Male",
    "F": "Female",
    "male": "Male",
    "female": "Female",
    "MALE": "Male",
    "FEMALE": "Female"
}

In [31]:
gender_before = (
    patients_clean[
        "Gender"
    ].copy()
)

In [32]:
patients_clean[
    "Gender"
] = (
    patients_clean[
        "Gender"
    ]
    .replace(gender_map)
)

In [33]:
gender_changed_count = (
    gender_before
    !=
    patients_clean[
        "Gender"
    ]
).sum()

print(
    "Gender values standardized:",
    gender_changed_count
)

Gender values standardized: 50


In [34]:
log_cleaning_action(
    "5.2",
    "patients",
    "Gender",
    "Standardized gender categories",
    gender_changed_count,
    "Non-standard categories detected during profiling"
)

In [35]:
valid_genders = [
    "Male",
    "Female",
    "Other",
    "Unknown"
]

print(
    "Remaining invalid Gender:",
    (
        ~patients_clean[
            "Gender"
        ].isin(valid_genders)
    ).sum()
)

Remaining invalid Gender: 0


### Recover missing Region

In [36]:
zip_region_map = (
    patients_clean[
        patients_clean[
            "Region"
        ].notna()
    ]
    .groupby(
        "ZipCode"
    )["Region"]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else np.nan
    )
    .to_dict()
)

In [37]:
missing_region_before = (
    patients_clean[
        "Region"
    ].isna()
)

print(
    "Missing Region before:",
    missing_region_before.sum()
)

Missing Region before: 100


In [38]:
patients_clean.loc[
    missing_region_before,
    "Region"
] = (
    patients_clean.loc[
        missing_region_before,
        "ZipCode"
    ]
    .map(zip_region_map)
)

In [39]:
patients_clean[
    "Region"
] = (
    patients_clean[
        "Region"
    ]
    .fillna("Unknown")
)

In [40]:
print(
    "Missing Region after:",
    patients_clean[
        "Region"
    ].isna().sum()
)

Missing Region after: 0


In [41]:
region_resolved_count = (
    missing_region_before.sum()
)

log_cleaning_action(
    "5.2",
    "patients",
    "Region",
    "Derived missing Region from ZipCode; unresolved mapped to Unknown",
    region_resolved_count,
    "Required for regional utilization and population reporting"
)

### Handle impossible BirthDates

In [42]:
study_end_date = pd.Timestamp(
    "2025-12-31"
)

invalid_birth_mask = (
    (
        patients_clean[
            "BirthDate"
        ] > study_end_date
    )
    |
    (
        patients_clean[
            "BirthDate"
        ]
        >
        patients_clean[
            "RegistrationDate"
        ]
    )
)

In [43]:
print(
    "Invalid BirthDates:",
    invalid_birth_mask.sum()
)

Invalid BirthDates: 33


In [44]:
patients_clean[
    "BirthDateValidFlag"
] = 1

patients_clean.loc[
    invalid_birth_mask,
    "BirthDateValidFlag"
] = 0

In [45]:
patients_clean.loc[
    invalid_birth_mask,
    "BirthDate"
] = pd.NaT

In [46]:
log_cleaning_action(
    "5.2",
    "patients",
    "BirthDate",
    "Invalid dates set to null and quality flag created",
    invalid_birth_mask.sum(),
    "Birth date could not be reliably inferred"
)

### Handle duplicate PatientIDs carefully

In [47]:
patients_clean.drop_duplicates()

,PatientID,BirthDate,Gender,Race,Ethnicity,ZipCode,State,Region,RegistrationDate,BirthDateValidFlag
0,PAT000001,2006-07-24,Male,White,Not Hispanic or Latino,22554,VA,Northern,2023-12-08,1
1,PAT000002,1953-04-07,Female,White,Not Hispanic or Latino,23834,VA,Southern,NaT,1
2,PAT000003,1994-03-09,Male,White,Not Hispanic or Latino,23606,VA,Eastern,NaT,1
3,PAT000004,2023-08-26,Female,Black or African American,Not Hispanic or Latino,22401,VA,Northern,NaT,1
4,PAT000005,2007-02-25,Female,White,Hispanic or Latino,23805,VA,Southern,2023-07-12,1
...,...,...,...,...,...,...,...,...,...,...
9995,PAT009996,1996-07-01,Male,Multiple,Not Hispanic or Latino,22911,VA,Western,NaT,1
9996,PAT009997,1990-03-04,Male,Unknown,Not Hispanic or Latino,22911,VA,Western,2024-12-03,1
9997,PAT009998,1968-04-08,Female,Black or African American,Not Hispanic or Latino,23225,VA,Central,2023-01-04,1
9998,PAT009999,2023-01-04,Male,White,Not Hispanic or Latino,23831,VA,Southern,NaT,1


In [48]:
duplicate_patient_mask = (
    patients_clean[
        "PatientID"
    ].duplicated(
        keep=False
    )
)

patient_duplicate_audit = (
    patients_clean[
        duplicate_patient_mask
    ]
    .sort_values(
        "PatientID"
    )
    .copy()
)

print(
    "Rows involved in duplicate IDs:",
    len(patient_duplicate_audit)
)

Rows involved in duplicate IDs: 40


In [49]:
patient_duplicate_audit.to_csv(
    QUARANTINE_PATH
    / "patient_duplicate_id_audit.csv",
    index=False
)

### Choose a deterministic canonical record

In [50]:
patients_clean = (
    patients_clean
    .sort_values(
        [
            "PatientID",
            "RegistrationDate"
        ]
    )
)

In [51]:
duplicate_rows_removed = (
    patients_clean[
        "PatientID"
    ].duplicated(
        keep="first"
    ).sum()
)

print(
    "Duplicate PatientID rows to remove:",
    duplicate_rows_removed
)

Duplicate PatientID rows to remove: 20


In [52]:
patients_clean = (
    patients_clean
    .drop_duplicates(
        subset=[
            "PatientID"
        ],
        keep="first"
    )
    .reset_index(
        drop=True
    )
)

In [53]:
log_cleaning_action(
    "5.2",
    "patients",
    "PatientID",
    "Canonicalized duplicate PatientIDs using earliest RegistrationDate",
    duplicate_rows_removed,
    "Primary key must be unique; conflicting source rows retained in quarantine audit"
)

### Validate cleaned patients

In [54]:
print(
    "Patient rows after cleaning:",
    len(patients_clean)
)

print(
    "Duplicate PatientID:",
    patients_clean[
        "PatientID"
    ].duplicated().sum()
)

print(
    "Missing PatientID:",
    patients_clean[
        "PatientID"
    ].isna().sum()
)

print(
    "Missing Region:",
    patients_clean[
        "Region"
    ].isna().sum()
)

print(
    "Invalid Gender:",
    (
        ~patients_clean[
            "Gender"
        ].isin(valid_genders)
    ).sum()
)

print(
    "Invalid BirthDates remaining:",
    (
        patients_clean[
            "BirthDate"
        ] > study_end_date
    ).sum()
)

Patient rows after cleaning: 9980
Duplicate PatientID: 0
Missing PatientID: 0
Missing Region: 0
Invalid Gender: 0
Invalid BirthDates remaining: 0


### Check an important consequence of patient deduplication

In [55]:
orphan_encounter_patients = (
    encounters_clean[
        encounters_clean[
            "PatientID"
        ].notna()
        &
        ~encounters_clean[
            "PatientID"
        ].isin(
            patients_clean[
                "PatientID"
            ]
        )
    ]
)

print(
    "Encounter rows referencing unavailable PatientIDs:",
    len(orphan_encounter_patients)
)

Encounter rows referencing unavailable PatientIDs: 176


### Save cleaned master tables

In [56]:
departments_clean.to_csv(
    CLEANED_DATA_PATH / "departments_clean.csv",
    index=False
)

payers_clean.to_csv(
    CLEANED_DATA_PATH / "payers_clean.csv",
    index=False
)

diagnoses_clean.to_csv(
    CLEANED_DATA_PATH / "diagnoses_clean.csv",
    index=False
)

procedures_clean.to_csv(
    CLEANED_DATA_PATH / "procedures_clean.csv",
    index=False
)

providers_clean.to_csv(
    CLEANED_DATA_PATH / "providers_clean.csv",
    index=False
)

patients_clean.to_csv(
    CLEANED_DATA_PATH / "patients_clean.csv",
    index=False
)

print(
    "Reference and patient cleaned datasets saved."
)

Reference and patient cleaned datasets saved.


### Inspect your cleaning audit log

In [57]:
cleaning_log_df = pd.DataFrame(
    cleaning_log
)

cleaning_log_df

,Step,Table,Field,Action,AffectedRows,Reason
0,5.2,patients,BirthDate / RegistrationDate,Converted text dates to datetime,10000,Required for temporal validation and age calcu...
1,5.2,patients,Gender,Standardized gender categories,50,Non-standard categories detected during profiling
2,5.2,patients,Region,Derived missing Region from ZipCode; unresolve...,100,Required for regional utilization and populati...
3,5.2,patients,BirthDate,Invalid dates set to null and quality flag cre...,33,Birth date could not be reliably inferred
4,5.2,patients,PatientID,Canonicalized duplicate PatientIDs using earli...,20,Primary key must be unique; conflicting source...


In [58]:
cleaning_log_df.to_csv(
    CLEANED_DATA_PATH
    / "cleaning_audit_log.csv",
    index=False
)

In [59]:
orphan_encounter_patients.to_csv(
    QUARANTINE_PATH
    / "encounters_unmapped_patient_audit.csv",
    index=False
)

In [60]:
log_cleaning_action(
    "5.2",
    "encounters",
    "PatientID",
    "Identified encounters with PatientIDs absent from canonical patient master",
    len(orphan_encounter_patients),
    "Patient master source corruption created orphan transactional references; preserve for Unknown Patient mapping during staging"
)

In [61]:
cleaning_log_df = pd.DataFrame(cleaning_log)

cleaning_log_df.to_csv(
    CLEANED_DATA_PATH
    / "cleaning_audit_log.csv",
    index=False
)

### Convert encounter dates and timestamps

In [449]:
print("Raw rows:", len(encounters_raw))
print("Clean rows:", len(encounters_clean))

print(
    "Same EncounterID order:",
    encounters_raw["EncounterID"]
    .reset_index(drop=True)
    .equals(
        encounters_clean["EncounterID"]
        .reset_index(drop=True)
    )
)

Raw rows: 90000
Clean rows: 90000
Same EncounterID order: True


In [450]:
# ------------------------------------------------------------
# Parse true datetime columns
# ------------------------------------------------------------

encounter_datetime_columns = [
    "EncounterDate",
    "ArrivalDateTime",
    "TriageDateTime",
    "ProviderStartDateTime"
]

for column in encounter_datetime_columns:

    encounters_clean[column] = pd.to_datetime(
        encounters_clean[column],
        errors="coerce"
    )


# ------------------------------------------------------------
# Reconstruct EncounterEndDateTime
#
# RAW EncounterEndDateTime is actually an elapsed-duration
# field in the format:
#
# Hours:Minutes.FractionOfMinute
#
# Example:
# 22:11.8 = 22 hours + 11.8 minutes
# ------------------------------------------------------------

raw_end_duration = (
    encounters_raw["EncounterEndDateTime"]
    .astype("string")
    .str.strip()
)

duration_parts = raw_end_duration.str.extract(
    r"^(?P<Hours>\d+):(?P<Minutes>\d{2})\.(?P<Fraction>\d+)$"
)

duration_hours = pd.to_numeric(
    duration_parts["Hours"],
    errors="coerce"
)

duration_minutes = pd.to_numeric(
    duration_parts["Minutes"],
    errors="coerce"
)

duration_fraction = pd.to_numeric(
    "0." + duration_parts["Fraction"],
    errors="coerce"
)

encounter_duration_minutes = (
    duration_hours * 60
    + duration_minutes
    + duration_fraction
)

encounters_clean["EncounterEndDateTime"] = (
    encounters_clean["ArrivalDateTime"]
    +
    pd.to_timedelta(
        encounter_duration_minutes,
        unit="m"
    )
)

print(
    "Duration values successfully parsed:",
    encounter_duration_minutes.notna().sum()
)

print(
    "Missing reconstructed encounter ends:",
    encounters_clean["EncounterEndDateTime"].isna().sum()
)

print(
    "Encounter ends before arrival:",
    (
        encounters_clean["EncounterEndDateTime"]
        <
        encounters_clean["ArrivalDateTime"]
    ).sum()
)

print(
    "Maximum reconstructed duration hours:",
    round(
        encounter_duration_minutes.max() / 60,
        2
    )
)

Duration values successfully parsed: 90000
Missing reconstructed encounter ends: 0
Encounter ends before arrival: 0
Maximum reconstructed duration hours: 60.0


In [63]:
log_cleaning_action(
    "5.3",
    "encounters",
    "Date/Time fields",
    "Converted encounter date/time fields to datetime",
    len(encounters_clean),
    "Required for temporal validation and operational KPI calculations"
)

### Convert cost to numeric

In [64]:
encounters_clean["EncounterCost"] = pd.to_numeric(
    encounters_clean["EncounterCost"],
    errors="coerce"
)

In [65]:
print(
    "Non-numeric / missing EncounterCost:",
    encounters_clean[
        "EncounterCost"
    ].isna().sum()
)

Non-numeric / missing EncounterCost: 0


### Standardize EncounterType

In [66]:
encounter_type_map = {
    "ER": "Emergency",
    "emergency": "Emergency",
    "Emergency": "Emergency",

    "Out Patient": "Outpatient",
    "Outpatient": "Outpatient",

    "PrimaryCare": "Primary Care",
    "Primary Care": "Primary Care",

    "SPECIALIST": "Specialist",
    "Specialist": "Specialist",

    "Urgent Care": "Urgent Care",
    "Inpatient": "Inpatient"
}

In [67]:
encounter_type_before = (
    encounters_clean[
        "EncounterType"
    ].copy()
)

In [68]:
encounters_clean[
    "EncounterType"
] = (
    encounters_clean[
        "EncounterType"
    ]
    .replace(
        encounter_type_map
    )
)

In [69]:
encounter_type_changed = (
    encounter_type_before
    !=
    encounters_clean[
        "EncounterType"
    ]
).sum()

print(
    "EncounterType values standardized:",
    encounter_type_changed
)

EncounterType values standardized: 450


In [70]:
log_cleaning_action(
    "5.3",
    "encounters",
    "EncounterType",
    "Standardized encounter type categories",
    encounter_type_changed,
    "Non-standard encounter categories identified during profiling"
)

In [71]:
valid_encounter_types = [
    "Emergency",
    "Inpatient",
    "Outpatient",
    "Urgent Care",
    "Primary Care",
    "Specialist"
]

remaining_invalid_types = (
    ~encounters_clean[
        "EncounterType"
    ].isin(
        valid_encounter_types
    )
).sum()

print(
    "Remaining invalid EncounterType:",
    remaining_invalid_types
)

Remaining invalid EncounterType: 0


### Handling the 176 orphan Patient references

In [72]:
encounters_clean[
    "PatientIDValidFlag"
] = (
    encounters_clean[
        "PatientID"
    ].isin(
        patients_clean[
            "PatientID"
        ]
    )
).astype(int)

In [73]:
print(
    "Invalid / unresolved PatientIDs:",
    (
        encounters_clean[
            "PatientIDValidFlag"
        ] == 0
    ).sum()
)

Invalid / unresolved PatientIDs: 176


In [74]:
encounters_clean[
    "PatientIDClean"
] = encounters_clean[
    "PatientID"
].where(
    encounters_clean[
        "PatientIDValidFlag"
    ] == 1,
    "UNKNOWN"
)

In [75]:
encounters_clean[
    "PatientIDClean"
].value_counts().head()

PatientIDClean
UNKNOWN      176
PAT003040     29
PAT008497     27
PAT007356     26
PAT001610     26
Name: count, dtype: int64

In [76]:
print(
    "Encounters mapped to UNKNOWN Patient:",
    (
        encounters_clean[
            "PatientIDClean"
        ] == "UNKNOWN"
    ).sum()
)

Encounters mapped to UNKNOWN Patient: 176


In [77]:
unmapped_patient_count = (
    encounters_clean[
        "PatientIDValidFlag"
    ] == 0
).sum()

log_cleaning_action(
    "5.3",
    "encounters",
    "PatientID",
    "Flagged unresolved PatientIDs and created staging-safe UNKNOWN mapping",
    unmapped_patient_count,
    "Preserve encounter transactions while preventing broken patient-dimension relationships"
)

### Handling invalid DepartmentIDs

In [78]:
invalid_department_mask = (
    ~encounters_clean[
        "DepartmentID"
    ].isin(
        departments_clean[
            "DepartmentID"
        ]
    )
)

print(
    "Invalid DepartmentIDs:",
    invalid_department_mask.sum()
)

Invalid DepartmentIDs: 270


In [79]:
encounters_clean[
    "DepartmentIDValidFlag"
] = (
    ~invalid_department_mask
).astype(int)

In [80]:
encounters_clean[
    "DepartmentIDClean"
] = encounters_clean[
    "DepartmentID"
].where(
    encounters_clean[
        "DepartmentIDValidFlag"
    ] == 1,
    "UNKNOWN"
)

In [81]:
print(
    "Encounters mapped to UNKNOWN Department:",
    (
        encounters_clean[
            "DepartmentIDClean"
        ] == "UNKNOWN"
    ).sum()
)

Encounters mapped to UNKNOWN Department: 270


In [82]:
log_cleaning_action(
    "5.3",
    "encounters",
    "DepartmentID",
    "Flagged invalid DepartmentIDs and created staging-safe UNKNOWN mapping",
    invalid_department_mask.sum(),
    "Preserve encounter volume while preventing invalid department joins"
)

### Validate ProviderID

In [83]:
provider_reference_valid = (
    encounters_clean[
        "ProviderID"
    ].isin(
        providers_clean[
            "ProviderID"
        ]
    )
)

In [84]:
encounters_clean[
    "ProviderIDValidFlag"
] = np.where(
    encounters_clean[
        "ProviderID"
    ].isna(),
    np.nan,
    provider_reference_valid.astype(int)
)

In [85]:
completed_missing_provider_mask = (
    (
        encounters_clean[
            "EncounterStatus"
        ] == "Completed"
    )
    &
    (
        encounters_clean[
            "ProviderID"
        ].isna()
    )
)

print(
    "Completed encounters missing provider:",
    completed_missing_provider_mask.sum()
)

Completed encounters missing provider: 900


### Create staging-safe ProviderID

In [86]:
encounters_clean[
    "ProviderIDClean"
] = encounters_clean[
    "ProviderID"
]

In [87]:
invalid_or_missing_provider_mask = (
    encounters_clean[
        "ProviderID"
    ].isna()
    |
    (
        ~encounters_clean[
            "ProviderID"
        ].isin(
            providers_clean[
                "ProviderID"
            ]
        )
    )
)

encounters_clean.loc[
    invalid_or_missing_provider_mask,
    "ProviderIDClean"
] = "UNKNOWN"

In [88]:
print(
    "Encounters using UNKNOWN Provider:",
    (
        encounters_clean[
            "ProviderIDClean"
        ] == "UNKNOWN"
    ).sum()
)

Encounters using UNKNOWN Provider: 1401


In [89]:
log_cleaning_action(
    "5.3",
    "encounters",
    "ProviderID",
    "Mapped unresolved provider references to UNKNOWN for staging",
    completed_missing_provider_mask.sum(),
    "Completed encounters require provider attribution for provider-level reporting"
)

### Handle negative encounter costs

In [90]:
negative_cost_mask = (
    encounters_clean[
        "EncounterCost"
    ] < 0
)

print(
    "Negative encounter costs:",
    negative_cost_mask.sum()
)

Negative encounter costs: 180


In [91]:
encounters_clean[
    "EncounterCostValidFlag"
] = (
    encounters_clean[
        "EncounterCost"
    ].ge(0)
    &
    encounters_clean[
        "EncounterCost"
    ].notna()
).astype(int)

In [92]:
encounters_clean[
    "EncounterCostClean"
] = encounters_clean[
    "EncounterCost"
].where(
    encounters_clean[
        "EncounterCostValidFlag"
    ] == 1,
    np.nan
)

In [93]:
print(
    "Invalid costs excluded from financial KPI field:",
    encounters_clean[
        "EncounterCostClean"
    ].isna().sum()
)

Invalid costs excluded from financial KPI field: 180


In [94]:
log_cleaning_action(
    "5.3",
    "encounters",
    "EncounterCost",
    "Flagged negative costs and excluded them from KPI-safe EncounterCostClean",
    negative_cost_mask.sum(),
    "Negative values cannot be assumed valid financial costs without source verification"
)

### Validate timestamp sequence

In [95]:
provider_before_arrival_mask = (
    encounters_clean[
        "ProviderStartDateTime"
    ].notna()
    &
    encounters_clean[
        "ArrivalDateTime"
    ].notna()
    &
    (
        encounters_clean[
            "ProviderStartDateTime"
        ]
        <
        encounters_clean[
            "ArrivalDateTime"
        ]
    )
)

print(
    "ProviderStart before Arrival:",
    provider_before_arrival_mask.sum()
)

ProviderStart before Arrival: 450


In [96]:
encounters_clean[
    "WaitTimeValidFlag"
] = 1

encounters_clean.loc[
    provider_before_arrival_mask,
    "WaitTimeValidFlag"
] = 0

In [97]:
missing_wait_timestamp_mask = (
    encounters_clean[
        "ArrivalDateTime"
    ].isna()
    |
    encounters_clean[
        "ProviderStartDateTime"
    ].isna()
)

In [98]:
encounters_clean.loc[
    missing_wait_timestamp_mask,
    "WaitTimeValidFlag"
] = 0

### Create WaitMinutes

In [99]:
encounters_clean[
    "WaitMinutes"
] = (
    encounters_clean[
        "ProviderStartDateTime"
    ]
    -
    encounters_clean[
        "ArrivalDateTime"
    ]
).dt.total_seconds() / 60

In [100]:
encounters_clean[
    "WaitMinutes"
] = encounters_clean[
    "WaitMinutes"
].where(
    encounters_clean[
        "WaitTimeValidFlag"
    ] == 1,
    np.nan
)

In [101]:
print(
    "Negative WaitMinutes remaining:",
    (
        encounters_clean[
            "WaitMinutes"
        ] < 0
    ).sum()
)

Negative WaitMinutes remaining: 0


In [102]:
log_cleaning_action(
    "5.3",
    "encounters",
    "ProviderStartDateTime / ArrivalDateTime",
    "Flagged invalid timestamp sequences and created KPI-safe WaitMinutes",
    provider_before_arrival_mask.sum(),
    "Prevent negative wait times from affecting operational KPIs"
)

### Validate encounter end timestamps

In [455]:
end_before_arrival_mask = (
    encounters_clean[
        "EncounterEndDateTime"
    ].notna()
    &
    encounters_clean[
        "ArrivalDateTime"
    ].notna()
    &
    (
        encounters_clean[
            "EncounterEndDateTime"
        ]
        <
        encounters_clean[
            "ArrivalDateTime"
        ]
    )
)

print(
    "Encounter end before arrival:",
    end_before_arrival_mask.sum()
)

Encounter end before arrival: 0


In [456]:
encounters_clean[
    "EncounterDurationValidFlag"
] = (
    ~end_before_arrival_mask
).astype(int)

In [457]:
print(
    encounters_clean[
        "EncounterDurationValidFlag"
    ].value_counts()
)

EncounterDurationValidFlag
1    90000
Name: count, dtype: int64


In [453]:
encounters_clean[
    "EncounterDurationHours"
] = (
    encounters_clean[
        "EncounterEndDateTime"
    ]
    -
    encounters_clean[
        "ArrivalDateTime"
    ]
).dt.total_seconds() / 3600

In [454]:
print(
    "Missing durations:",
    encounters_clean["EncounterDurationHours"].isna().sum()
)

print(
    "Negative durations:",
    (
        encounters_clean["EncounterDurationHours"] < 0
    ).sum()
)

print(
    "Minimum duration hours:",
    encounters_clean["EncounterDurationHours"].min()
)

print(
    "Maximum duration hours:",
    encounters_clean["EncounterDurationHours"].max()
)

print(
    "\nDuration percentiles:"
)

print(
    encounters_clean["EncounterDurationHours"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
    .round(2)
)

Missing durations: 0
Negative durations: 0
Minimum duration hours: 0.0
Maximum duration hours: 59.998333333333335

Duration percentiles:
0.50    30.02
0.75    45.01
0.90    54.06
0.95    57.01
0.99    59.39
1.00    60.00
Name: EncounterDurationHours, dtype: float64


In [106]:
encounters_clean[
    "EncounterDurationHours"
] = encounters_clean[
    "EncounterDurationHours"
].where(
    encounters_clean[
        "EncounterDurationValidFlag"
    ] == 1,
    np.nan
)

### Check EncounterDate consistency

In [107]:
encounter_date_mismatch_mask = (
    encounters_clean[
        "EncounterDate"
    ].dt.date
    !=
    encounters_clean[
        "ArrivalDateTime"
    ].dt.date
)

print(
    "EncounterDate / ArrivalDate mismatches:",
    encounter_date_mismatch_mask.sum()
)

EncounterDate / ArrivalDate mismatches: 0


### Validate encounter status

In [108]:
valid_encounter_statuses = [
    "Completed",
    "Cancelled",
    "Left Without Being Seen",
    "Transferred"
]

invalid_status_mask = (
    ~encounters_clean[
        "EncounterStatus"
    ].isin(
        valid_encounter_statuses
    )
)

print(
    "Invalid EncounterStatus:",
    invalid_status_mask.sum()
)

Invalid EncounterStatus: 0


### Create an overall quality flag

In [109]:
encounters_clean[
    "EncounterQualityFlag"
] = np.where(
    (
        (encounters_clean["PatientIDValidFlag"] == 1)
        &
        (encounters_clean["DepartmentIDValidFlag"] == 1)
        &
        (encounters_clean["EncounterCostValidFlag"] == 1)
        &
        (
            ~provider_before_arrival_mask
        )
    ),
    1,
    0
)

### Create a cleaning summary

In [110]:
encounter_cleaning_summary = pd.DataFrame([
    {
        "Issue": "Unresolved PatientID",
        "AffectedRows": (
            encounters_clean[
                "PatientIDValidFlag"
            ] == 0
        ).sum()
    },
    {
        "Issue": "Invalid DepartmentID",
        "AffectedRows": (
            encounters_clean[
                "DepartmentIDValidFlag"
            ] == 0
        ).sum()
    },
    {
        "Issue": "Completed Missing Provider",
        "AffectedRows": (
            completed_missing_provider_mask
        ).sum()
    },
    {
        "Issue": "Negative Encounter Cost",
        "AffectedRows": (
            negative_cost_mask
        ).sum()
    },
    {
        "Issue": "Invalid Wait-Time Sequence",
        "AffectedRows": (
            provider_before_arrival_mask
        ).sum()
    },
    {
        "Issue": "Invalid EncounterType Remaining",
        "AffectedRows": (
            ~encounters_clean[
                "EncounterType"
            ].isin(valid_encounter_types)
        ).sum()
    }
])

encounter_cleaning_summary

,Issue,AffectedRows
0,Unresolved PatientID,176
1,Invalid DepartmentID,270
2,Completed Missing Provider,900
3,Negative Encounter Cost,180
4,Invalid Wait-Time Sequence,450
5,Invalid EncounterType Remaining,0


### Validate key outcomes

In [111]:
print("=" * 60)
print("ENCOUNTER ETL VALIDATION")
print("=" * 60)

print(
    "Rows:",
    len(encounters_clean)
)

print(
    "Duplicate EncounterID:",
    encounters_clean[
        "EncounterID"
    ].duplicated().sum()
)

print(
    "Missing EncounterID:",
    encounters_clean[
        "EncounterID"
    ].isna().sum()
)

print(
    "Invalid EncounterType:",
    (
        ~encounters_clean[
            "EncounterType"
        ].isin(valid_encounter_types)
    ).sum()
)

print(
    "Negative EncounterCostClean:",
    (
        encounters_clean[
            "EncounterCostClean"
        ] < 0
    ).sum()
)

print(
    "Negative WaitMinutes:",
    (
        encounters_clean[
            "WaitMinutes"
        ] < 0
    ).sum()
)

print(
    "Invalid staging Patient references:",
    encounters_clean[
        "PatientIDClean"
    ].isna().sum()
)

print(
    "Invalid staging Department references:",
    encounters_clean[
        "DepartmentIDClean"
    ].isna().sum()
)

ENCOUNTER ETL VALIDATION
Rows: 90000
Duplicate EncounterID: 0
Missing EncounterID: 0
Invalid EncounterType: 0
Negative EncounterCostClean: 0
Negative WaitMinutes: 0
Invalid staging Patient references: 0
Invalid staging Department references: 0


In [440]:
valid_duration_mask = (
    encounters_clean[
        "ArrivalDateTime"
    ].notna()
    &
    encounters_clean[
        "EncounterEndDateTime"
    ].notna()
    &
    (
        encounters_clean[
            "EncounterEndDateTime"
        ]
        >=
        encounters_clean[
            "ArrivalDateTime"
        ]
    )
)

encounters_clean[
    "EncounterDurationValidFlag"
] = valid_duration_mask.astype(int)

encounters_clean[
    "EncounterDurationHours"
] = np.nan

encounters_clean.loc[
    valid_duration_mask,
    "EncounterDurationHours"
] = (
    encounters_clean.loc[
        valid_duration_mask,
        "EncounterEndDateTime"
    ]
    -
    encounters_clean.loc[
        valid_duration_mask,
        "ArrivalDateTime"
    ]
).dt.total_seconds() / 3600

### Save cleaned encounters

In [ ]:
encounters_clean.to_csv(
    CLEANED_DATA_PATH
    / "encounters_clean.csv",
    index=False
)

print(
    "encounters_clean.csv saved successfully."
)

encounters_clean.csv saved successfully.


In [113]:
encounter_cleaning_summary.to_csv(
    CLEANED_DATA_PATH
    / "encounter_cleaning_summary.csv",
    index=False
)

### Update your audit log

In [114]:
cleaning_log_df = pd.DataFrame(
    cleaning_log
)

cleaning_log_df.to_csv(
    CLEANED_DATA_PATH
    / "cleaning_audit_log.csv",
    index=False
)

cleaning_log_df.tail(10)

,Step,Table,Field,Action,AffectedRows,Reason
3,5.2,patients,BirthDate,Invalid dates set to null and quality flag cre...,33,Birth date could not be reliably inferred
4,5.2,patients,PatientID,Canonicalized duplicate PatientIDs using earli...,20,Primary key must be unique; conflicting source...
5,5.2,encounters,PatientID,Identified encounters with PatientIDs absent f...,176,Patient master source corruption created orpha...
6,5.3,encounters,Date/Time fields,Converted encounter date/time fields to datetime,90000,Required for temporal validation and operation...
7,5.3,encounters,EncounterType,Standardized encounter type categories,450,Non-standard encounter categories identified d...
8,5.3,encounters,PatientID,Flagged unresolved PatientIDs and created stag...,176,Preserve encounter transactions while preventi...
9,5.3,encounters,DepartmentID,Flagged invalid DepartmentIDs and created stag...,270,Preserve encounter volume while preventing inv...
10,5.3,encounters,ProviderID,Mapped unresolved provider references to UNKNO...,900,Completed encounters require provider attribut...
11,5.3,encounters,EncounterCost,Flagged negative costs and excluded them from ...,180,Negative values cannot be assumed valid financ...
12,5.3,encounters,ProviderStartDateTime / ArrivalDateTime,Flagged invalid timestamp sequences and create...,450,Prevent negative wait times from affecting ope...


### Clean encounter_diagnoses + encounter_procedures

#### Part A — encounter_diagnoses
#### Standardize data types

In [115]:
encounter_diagnoses_clean[
    "PresentOnAdmissionFlag"
] = pd.to_numeric(
    encounter_diagnoses_clean[
        "PresentOnAdmissionFlag"
    ],
    errors="coerce"
).astype("Int64")

In [116]:
print(
    "Rows:",
    len(encounter_diagnoses_clean)
)

print(
    "Duplicate EncounterDiagnosisID:",
    encounter_diagnoses_clean[
        "EncounterDiagnosisID"
    ].duplicated().sum()
)

Rows: 153595
Duplicate EncounterDiagnosisID: 0


### Validate EncounterID

In [117]:
invalid_dx_encounter_mask = (
    ~encounter_diagnoses_clean[
        "EncounterID"
    ].isin(
        encounters_clean[
            "EncounterID"
        ]
    )
)

print(
    "Invalid EncounterID references:",
    invalid_dx_encounter_mask.sum()
)

Invalid EncounterID references: 0


### Identify invalid DiagnosisIDs

In [118]:
invalid_diagnosis_mask = (
    ~encounter_diagnoses_clean[
        "DiagnosisID"
    ].isin(
        diagnoses_clean[
            "DiagnosisID"
        ]
    )
)

print(
    "Invalid DiagnosisIDs:",
    invalid_diagnosis_mask.sum()
)

Invalid DiagnosisIDs: 459


In [119]:
encounter_diagnoses_clean[
    "DiagnosisIDValidFlag"
] = (
    ~invalid_diagnosis_mask
).astype(int)

In [120]:
encounter_diagnoses_clean[
    "DiagnosisIDClean"
] = encounter_diagnoses_clean[
    "DiagnosisID"
].where(
    encounter_diagnoses_clean[
        "DiagnosisIDValidFlag"
    ] == 1,
    "UNKNOWN"
)

In [121]:
print(
    "Rows mapped to UNKNOWN Diagnosis:",
    (
        encounter_diagnoses_clean[
            "DiagnosisIDClean"
        ] == "UNKNOWN"
    ).sum()
)

Rows mapped to UNKNOWN Diagnosis: 459


In [122]:
log_cleaning_action(
    "5.4",
    "encounter_diagnoses",
    "DiagnosisID",
    "Flagged invalid DiagnosisIDs and mapped unresolved values to UNKNOWN",
    invalid_diagnosis_mask.sum(),
    "Preserve encounter diagnosis records while preventing invalid dimension joins"
)

### Audit duplicate encounter-diagnosis assignments

In [123]:
valid_dx_rows_mask = (
    encounter_diagnoses_clean[
        "DiagnosisIDValidFlag"
    ] == 1
)

duplicate_dx_pair_mask = (
    encounter_diagnoses_clean[
        valid_dx_rows_mask
    ]
    .duplicated(
        subset=[
            "EncounterID",
            "DiagnosisID"
        ],
        keep=False
    )
)

In [124]:
duplicate_dx_audit = (
    encounter_diagnoses_clean[
        valid_dx_rows_mask
    ]
    .loc[
        duplicate_dx_pair_mask
    ]
    .sort_values(
        [
            "EncounterID",
            "DiagnosisID"
        ]
    )
    .copy()
)

print(
    "Rows involved in valid duplicate diagnosis pairs:",
    len(duplicate_dx_audit)
)

Rows involved in valid duplicate diagnosis pairs: 916


In [125]:
duplicate_dx_audit.to_csv(
    QUARANTINE_PATH
    / "encounter_diagnosis_duplicate_audit.csv",
    index=False
)

### Remove confirmed business-key duplicates

In [126]:
valid_duplicate_remove_mask = (
    encounter_diagnoses_clean[
        "DiagnosisIDValidFlag"
    ].eq(1)
    &
    encounter_diagnoses_clean.duplicated(
        subset=[
            "EncounterID",
            "DiagnosisID"
        ],
        keep="first"
    )
)

diagnosis_duplicates_removed = (
    valid_duplicate_remove_mask.sum()
)

print(
    "Duplicate diagnosis assignments to remove:",
    diagnosis_duplicates_removed
)

Duplicate diagnosis assignments to remove: 458


In [127]:
encounter_diagnoses_clean = (
    encounter_diagnoses_clean[
        ~valid_duplicate_remove_mask
    ]
    .reset_index(drop=True)
)

In [128]:
log_cleaning_action(
    "5.4",
    "encounter_diagnoses",
    "EncounterID + DiagnosisID",
    "Removed confirmed duplicate valid encounter-diagnosis assignments",
    diagnosis_duplicates_removed,
    "Duplicate valid diagnosis assignments would overstate diagnosis prevalence and utilization"
)

### Validate DiagnosisType

In [129]:
valid_diagnosis_types = [
    "Primary",
    "Secondary"
]

invalid_diagnosis_type_mask = (
    ~encounter_diagnoses_clean[
        "DiagnosisType"
    ].isin(
        valid_diagnosis_types
    )
)

print(
    "Invalid DiagnosisType:",
    invalid_diagnosis_type_mask.sum()
)

Invalid DiagnosisType: 0


### Check one Primary diagnosis per encounter

In [130]:
primary_diagnosis_counts = (
    encounter_diagnoses_clean[
        encounter_diagnoses_clean[
            "DiagnosisType"
        ] == "Primary"
    ]
    .groupby(
        "EncounterID"
    )
    .size()
)

In [131]:
multiple_primary_count = (
    primary_diagnosis_counts > 1
).sum()

print(
    "Encounters with multiple Primary diagnoses:",
    multiple_primary_count
)

Encounters with multiple Primary diagnoses: 0


In [132]:
encounters_with_diagnosis = (
    encounter_diagnoses_clean[
        "EncounterID"
    ].unique()
)

encounters_with_primary = (
    encounter_diagnoses_clean.loc[
        encounter_diagnoses_clean[
            "DiagnosisType"
        ] == "Primary",
        "EncounterID"
    ].unique()
)

no_primary_count = len(
    set(encounters_with_diagnosis)
    -
    set(encounters_with_primary)
)

print(
    "Encounters with diagnosis rows but no Primary diagnosis:",
    no_primary_count
)

Encounters with diagnosis rows but no Primary diagnosis: 0


In [133]:
invalid_poa_mask = (
    encounter_diagnoses_clean[
        "PresentOnAdmissionFlag"
    ].notna()
    &
    ~encounter_diagnoses_clean[
        "PresentOnAdmissionFlag"
    ].isin([0, 1])
)

print(
    "Invalid PresentOnAdmissionFlag:",
    invalid_poa_mask.sum()
)

Invalid PresentOnAdmissionFlag: 0


### Part B — encounter_procedures
### Convert procedure fields

In [134]:
encounter_procedures_clean[
    "ProcedureDate"
] = pd.to_datetime(
    encounter_procedures_clean[
        "ProcedureDate"
    ],
    errors="coerce"
)

encounter_procedures_clean[
    "ProcedureCost"
] = pd.to_numeric(
    encounter_procedures_clean[
        "ProcedureCost"
    ],
    errors="coerce"
)

In [135]:
print(
    "Rows:",
    len(encounter_procedures_clean)
)

print(
    "Duplicate EncounterProcedureID:",
    encounter_procedures_clean[
        "EncounterProcedureID"
    ].duplicated().sum()
)

Rows: 70419
Duplicate EncounterProcedureID: 0


### Validate ProcedureID

In [136]:
invalid_procedure_mask = (
    ~encounter_procedures_clean[
        "ProcedureID"
    ].isin(
        procedures_clean[
            "ProcedureID"
        ]
    )
)

print(
    "Invalid ProcedureIDs:",
    invalid_procedure_mask.sum()
)

Invalid ProcedureIDs: 140


In [137]:
encounter_procedures_clean[
    "ProcedureIDValidFlag"
] = (
    ~invalid_procedure_mask
).astype(int)

encounter_procedures_clean[
    "ProcedureIDClean"
] = encounter_procedures_clean[
    "ProcedureID"
].where(
    encounter_procedures_clean[
        "ProcedureIDValidFlag"
    ] == 1,
    "UNKNOWN"
)

In [138]:
log_cleaning_action(
    "5.4",
    "encounter_procedures",
    "ProcedureID",
    "Flagged invalid ProcedureIDs and mapped unresolved values to UNKNOWN",
    invalid_procedure_mask.sum(),
    "Preserve procedure transactions while preventing invalid procedure-dimension joins"
)

### Handle negative ProcedureCost

In [139]:
negative_procedure_cost_mask = (
    encounter_procedures_clean[
        "ProcedureCost"
    ] < 0
)

print(
    "Negative ProcedureCost:",
    negative_procedure_cost_mask.sum()
)

Negative ProcedureCost: 140


In [140]:
encounter_procedures_clean[
    "ProcedureCostValidFlag"
] = (
    encounter_procedures_clean[
        "ProcedureCost"
    ].notna()
    &
    encounter_procedures_clean[
        "ProcedureCost"
    ].ge(0)
).astype(int)

In [141]:
encounter_procedures_clean[
    "ProcedureCostClean"
] = encounter_procedures_clean[
    "ProcedureCost"
].where(
    encounter_procedures_clean[
        "ProcedureCostValidFlag"
    ] == 1,
    np.nan
)

In [142]:
print(
    "Negative ProcedureCostClean:",
    (
        encounter_procedures_clean[
            "ProcedureCostClean"
        ] < 0
    ).sum()
)

Negative ProcedureCostClean: 0


In [143]:
log_cleaning_action(
    "5.4",
    "encounter_procedures",
    "ProcedureCost",
    "Flagged negative costs and created KPI-safe ProcedureCostClean",
    negative_procedure_cost_mask.sum(),
    "Negative costs cannot be assumed valid without source verification"
)

### Validate procedure dates against encounters

In [155]:
procedure_date_check = (
    encounter_procedures_clean
    .merge(
        encounters_clean[
            [
                "EncounterID",
                "ArrivalDateTime",
                "EncounterEndDateTime"
            ]
        ],
        on="EncounterID",
        how="left"
    )
)

In [156]:
procedure_date_check[
    "ProcedureDateOnly"
] = (
    pd.to_datetime(
        procedure_date_check[
            "ProcedureDate"
        ]
    )
    .dt.normalize()
)

procedure_date_check[
    "ArrivalDateOnly"
] = (
    pd.to_datetime(
        procedure_date_check[
            "ArrivalDateTime"
        ]
    )
    .dt.normalize()
)

procedure_date_check[
    "EncounterEndDateOnly"
] = (
    pd.to_datetime(
        procedure_date_check[
            "EncounterEndDateTime"
        ]
    )
    .dt.normalize()
)

In [157]:
procedure_before_encounter_mask = (
    procedure_date_check[
        "ProcedureDateOnly"
    ].notna()
    &
    procedure_date_check[
        "ArrivalDateOnly"
    ].notna()
    &
    (
        procedure_date_check[
            "ProcedureDateOnly"
        ]
        <
        procedure_date_check[
            "ArrivalDateOnly"
        ]
    )
)

In [158]:
procedure_after_encounter_mask = (
    procedure_date_check[
        "ProcedureDateOnly"
    ].notna()
    &
    procedure_date_check[
        "EncounterEndDateOnly"
    ].notna()
    &
    (
        procedure_date_check[
            "ProcedureDateOnly"
        ]
        >
        procedure_date_check[
            "EncounterEndDateOnly"
        ]
    )
)

In [159]:
print(
    "Procedure before encounter:",
    procedure_before_encounter_mask.sum()
)

print(
    "Procedure after encounter:",
    procedure_after_encounter_mask.sum()
)

Procedure before encounter: 0
Procedure after encounter: 0


### Create procedure date validity flag

In [160]:
procedure_date_invalid_series = (
    procedure_before_encounter_mask
    |
    procedure_after_encounter_mask
)

In [161]:
encounter_procedures_clean[
    "ProcedureDateValidFlag"
] = (
    ~procedure_date_invalid_series
).astype(int).values

### Investigate potential duplicate procedures

In [162]:
potential_duplicate_procedure_mask = (
    encounter_procedures_clean
    .duplicated(
        subset=[
            "EncounterID",
            "ProcedureID",
            "ProcedureDate"
        ],
        keep=False
    )
)

print(
    "Rows involved in potential duplicate procedure events:",
    potential_duplicate_procedure_mask.sum()
)

Rows involved in potential duplicate procedure events: 0


In [163]:
potential_duplicate_procedures = (
    encounter_procedures_clean[
        potential_duplicate_procedure_mask
    ]
    .sort_values(
        [
            "EncounterID",
            "ProcedureDate",
            "ProcedureID"
        ]
    )
)

potential_duplicate_procedures.to_csv(
    QUARANTINE_PATH
    / "potential_duplicate_procedure_audit.csv",
    index=False
)

In [164]:
encounter_procedures_clean[
    "PotentialDuplicateFlag"
] = (
    potential_duplicate_procedure_mask
).astype(int)

### Validate the two cleaned tables

In [165]:
print("=" * 65)
print("ENCOUNTER DIAGNOSIS ETL VALIDATION")
print("=" * 65)

print(
    "Rows:",
    len(encounter_diagnoses_clean)
)

print(
    "Duplicate EncounterDiagnosisID:",
    encounter_diagnoses_clean[
        "EncounterDiagnosisID"
    ].duplicated().sum()
)

print(
    "Remaining valid duplicate diagnosis pairs:",
    encounter_diagnoses_clean[
        encounter_diagnoses_clean[
            "DiagnosisIDValidFlag"
        ] == 1
    ].duplicated(
        subset=[
            "EncounterID",
            "DiagnosisID"
        ]
    ).sum()
)

print(
    "Multiple Primary diagnoses:",
    (
        encounter_diagnoses_clean[
            encounter_diagnoses_clean[
                "DiagnosisType"
            ] == "Primary"
        ]
        .groupby("EncounterID")
        .size()
        .gt(1)
        .sum()
    )
)

ENCOUNTER DIAGNOSIS ETL VALIDATION
Rows: 153137
Duplicate EncounterDiagnosisID: 0
Remaining valid duplicate diagnosis pairs: 0
Multiple Primary diagnoses: 0


In [166]:
print("=" * 65)
print("ENCOUNTER PROCEDURE ETL VALIDATION")
print("=" * 65)

print(
    "Rows:",
    len(encounter_procedures_clean)
)

print(
    "Duplicate EncounterProcedureID:",
    encounter_procedures_clean[
        "EncounterProcedureID"
    ].duplicated().sum()
)

print(
    "Negative ProcedureCostClean:",
    (
        encounter_procedures_clean[
            "ProcedureCostClean"
        ] < 0
    ).sum()
)

print(
    "Invalid Procedure Dates:",
    (
        encounter_procedures_clean[
            "ProcedureDateValidFlag"
        ] == 0
    ).sum()
)

ENCOUNTER PROCEDURE ETL VALIDATION
Rows: 70419
Duplicate EncounterProcedureID: 0
Negative ProcedureCostClean: 0
Invalid Procedure Dates: 0


In [167]:
encounter_diagnoses_clean.to_csv(
    CLEANED_DATA_PATH
    / "encounter_diagnoses_clean.csv",
    index=False
)

encounter_procedures_clean.to_csv(
    CLEANED_DATA_PATH
    / "encounter_procedures_clean.csv",
    index=False
)

print(
    "Encounter diagnosis and procedure cleaned files saved."
)

Encounter diagnosis and procedure cleaned files saved.


In [168]:
cleaning_log_df = pd.DataFrame(
    cleaning_log
)

cleaning_log_df.to_csv(
    CLEANED_DATA_PATH
    / "cleaning_audit_log.csv",
    index=False
)

cleaning_log_df.tail(10)

,Step,Table,Field,Action,AffectedRows,Reason
7,5.3,encounters,EncounterType,Standardized encounter type categories,450,Non-standard encounter categories identified d...
8,5.3,encounters,PatientID,Flagged unresolved PatientIDs and created stag...,176,Preserve encounter transactions while preventi...
9,5.3,encounters,DepartmentID,Flagged invalid DepartmentIDs and created stag...,270,Preserve encounter volume while preventing inv...
10,5.3,encounters,ProviderID,Mapped unresolved provider references to UNKNO...,900,Completed encounters require provider attribut...
11,5.3,encounters,EncounterCost,Flagged negative costs and excluded them from ...,180,Negative values cannot be assumed valid financ...
12,5.3,encounters,ProviderStartDateTime / ArrivalDateTime,Flagged invalid timestamp sequences and create...,450,Prevent negative wait times from affecting ope...
13,5.4,encounter_diagnoses,DiagnosisID,Flagged invalid DiagnosisIDs and mapped unreso...,459,Preserve encounter diagnosis records while pre...
14,5.4,encounter_diagnoses,EncounterID + DiagnosisID,Removed confirmed duplicate valid encounter-di...,458,Duplicate valid diagnosis assignments would ov...
15,5.4,encounter_procedures,ProcedureID,Flagged invalid ProcedureIDs and mapped unreso...,140,Preserve procedure transactions while preventi...
16,5.4,encounter_procedures,ProcedureCost,Flagged negative costs and created KPI-safe Pr...,140,Negative costs cannot be assumed valid without...


### Clean admissions
### Convert admission date fields

In [169]:
admission_datetime_columns = [
    "AdmissionDateTime",
    "DischargeDateTime",
    "FollowUpDate"
]

for column in admission_datetime_columns:
    admissions_clean[column] = pd.to_datetime(
        admissions_clean[column],
        errors="coerce"
    )

In [170]:
admissions_clean[
    "FollowUpRequiredFlag"
] = pd.to_numeric(
    admissions_clean[
        "FollowUpRequiredFlag"
    ],
    errors="coerce"
).astype("Int64")

admissions_clean[
    "FollowUpCompletedFlag"
] = pd.to_numeric(
    admissions_clean[
        "FollowUpCompletedFlag"
    ],
    errors="coerce"
).astype("Int64")

In [171]:
print(
    admissions_clean[
        admission_datetime_columns
    ].dtypes
)

print(
    admissions_clean[
        [
            "FollowUpRequiredFlag",
            "FollowUpCompletedFlag"
        ]
    ].dtypes
)

AdmissionDateTime    datetime64[us]
DischargeDateTime    datetime64[us]
FollowUpDate         datetime64[us]
dtype: object
FollowUpRequiredFlag     Int64
FollowUpCompletedFlag    Int64
dtype: object


In [172]:
log_cleaning_action(
    "5.5",
    "admissions",
    "Admission/Discharge/Follow-Up dates",
    "Converted admission date fields to datetime",
    len(admissions_clean),
    "Required for LOS, follow-up, and readmission analysis"
)

### Validate the primary key

In [173]:
print(
    "Rows:",
    len(admissions_clean)
)

print(
    "Missing AdmissionID:",
    admissions_clean[
        "AdmissionID"
    ].isna().sum()
)

print(
    "Duplicate AdmissionID:",
    admissions_clean[
        "AdmissionID"
    ].duplicated().sum()
)

Rows: 17740
Missing AdmissionID: 0
Duplicate AdmissionID: 0


### Validate EncounterID

In [174]:
invalid_admission_encounter_mask = (
    ~admissions_clean[
        "EncounterID"
    ].isin(
        encounters_clean[
            "EncounterID"
        ]
    )
)

print(
    "Invalid admission EncounterID:",
    invalid_admission_encounter_mask.sum()
)

Invalid admission EncounterID: 0


In [175]:
admissions_clean[
    "EncounterIDValidFlag"
] = (
    ~invalid_admission_encounter_mask
).astype(int)

### Validate patient consistency with encounter

In [176]:
admission_encounter_check = (
    admissions_clean
    .merge(
        encounters_clean[
            [
                "EncounterID",
                "PatientID",
                "DepartmentID",
                "ArrivalDateTime",
                "EncounterEndDateTime"
            ]
        ],
        on="EncounterID",
        how="left",
        suffixes=(
            "_Admission",
            "_Encounter"
        )
    )
)

In [177]:
admission_patient_mismatch_mask = (
    admission_encounter_check[
        "PatientID_Admission"
    ]
    !=
    admission_encounter_check[
        "PatientID_Encounter"
    ]
)

print(
    "Admission / Encounter patient mismatches:",
    admission_patient_mismatch_mask.sum()
)

Admission / Encounter patient mismatches: 0


### Validate department consistency

In [178]:
admission_department_mismatch_mask = (
    admission_encounter_check[
        "DepartmentID_Admission"
    ]
    !=
    admission_encounter_check[
        "DepartmentID_Encounter"
    ]
)

print(
    "Admission / Encounter department mismatches:",
    admission_department_mismatch_mask.sum()
)

Admission / Encounter department mismatches: 54


In [179]:
mismatch_detail = (
    admission_encounter_check[
        admission_department_mismatch_mask
    ]
    .copy()
)

mismatch_detail[
    "AdmissionDeptValid"
] = mismatch_detail[
    "DepartmentID_Admission"
].isin(
    departments_clean[
        "DepartmentID"
    ]
)

mismatch_detail[
    "EncounterDeptValid"
] = mismatch_detail[
    "DepartmentID_Encounter"
].isin(
    departments_clean[
        "DepartmentID"
    ]
)

print(
    mismatch_detail[
        [
            "AdmissionDeptValid",
            "EncounterDeptValid"
        ]
    ]
    .value_counts()
)

AdmissionDeptValid  EncounterDeptValid
True                False                 54
Name: count, dtype: int64


In [180]:
mismatch_detail[
    [
        "EncounterID",
        "DepartmentID_Admission",
        "DepartmentID_Encounter"
    ]
].head(20)

,EncounterID,DepartmentID_Admission,DepartmentID_Encounter
1901,ENC009932,DEP002,DEP999
2143,ENC011298,DEP013,DEP999
3262,ENC016860,DEP008,DEP999
3431,ENC017674,DEP002,DEP999
3701,ENC019281,DEP013,DEP999
3994,ENC020745,DEP013,DEP999
4312,ENC022423,DEP008,DEP999
4686,ENC024184,DEP013,DEP999
5353,ENC027492,DEP008,DEP999
5419,ENC027811,DEP013,DEP999


In [181]:
valid_admission_department_mask = (
    admissions_clean[
        "DepartmentID"
    ].isin(
        departments_clean[
            "DepartmentID"
        ]
    )
)

admission_department_lookup = (
    admissions_clean[
        valid_admission_department_mask
    ]
    .drop_duplicates(
        subset="EncounterID"
    )
    .set_index(
        "EncounterID"
    )["DepartmentID"]
)

In [182]:
recoverable_encounter_department_mask = (
    encounters_clean[
        "DepartmentIDValidFlag"
    ].eq(0)
    &
    encounters_clean[
        "EncounterID"
    ].isin(
        admission_department_lookup.index
    )
)

print(
    "Invalid encounter departments recoverable from admissions:",
    recoverable_encounter_department_mask.sum()
)

Invalid encounter departments recoverable from admissions: 54


In [183]:
encounters_clean.loc[
    recoverable_encounter_department_mask,
    "DepartmentIDClean"
] = (
    encounters_clean.loc[
        recoverable_encounter_department_mask,
        "EncounterID"
    ]
    .map(
        admission_department_lookup
    )
)

In [184]:
encounters_clean[
    "DepartmentIDRecoveredFlag"
] = 0

encounters_clean.loc[
    recoverable_encounter_department_mask,
    "DepartmentIDRecoveredFlag"
] = 1

In [185]:
encounters_clean[
    "DepartmentIDResolvedFlag"
] = (
    encounters_clean[
        "DepartmentIDClean"
    ].isin(
        departments_clean[
            "DepartmentID"
        ]
    )
).astype(int)

In [186]:
print(
    "Source DepartmentID invalid:",
    (
        encounters_clean[
            "DepartmentIDValidFlag"
        ] == 0
    ).sum()
)

print(
    "Recovered from admissions:",
    encounters_clean[
        "DepartmentIDRecoveredFlag"
    ].sum()
)

print(
    "Still mapped to UNKNOWN:",
    (
        encounters_clean[
            "DepartmentIDClean"
        ] == "UNKNOWN"
    ).sum()
)

Source DepartmentID invalid: 270
Recovered from admissions: 54
Still mapped to UNKNOWN: 216


In [187]:
log_cleaning_action(
    "5.5",
    "encounters",
    "DepartmentIDClean",
    "Recovered invalid encounter departments using linked admission DepartmentID",
    recoverable_encounter_department_mask.sum(),
    "Admission record provided a valid department for the same inpatient EncounterID"
)

In [188]:
encounters_clean.to_csv(
    CLEANED_DATA_PATH
    / "encounters_clean.csv",
    index=False
)

In [189]:
admission_clean_department_check = (
    admissions_clean
    .merge(
        encounters_clean[
            [
                "EncounterID",
                "DepartmentIDClean"
            ]
        ],
        on="EncounterID",
        how="left"
    )
)

remaining_department_mismatches = (
    admission_clean_department_check[
        "DepartmentID"
    ]
    !=
    admission_clean_department_check[
        "DepartmentIDClean"
    ]
)

print(
    "Admission / cleaned encounter department mismatches:",
    remaining_department_mismatches.sum()
)

Admission / cleaned encounter department mismatches: 0


### Recover missing DischargeDateTime

In [190]:
missing_discharge_mask = (
    admissions_clean["DischargeDateTime"].isna()
)

print(
    "Missing DischargeDateTime:",
    missing_discharge_mask.sum()
)

Missing DischargeDateTime: 124


In [462]:
encounter_end_lookup = (
    encounters_clean
    .set_index("EncounterID")[
        "EncounterEndDateTime"
    ]
)

In [463]:
missing_discharge_before = (
    admissions_clean[
        "DischargeDateTime"
    ].isna().sum()
)

print(
    "Missing discharge before recovery:",
    missing_discharge_before
)

Missing discharge before recovery: 0


In [464]:
admissions_clean.loc[
    missing_discharge_mask,
    "DischargeDateTime"
] = (
    admissions_clean.loc[
        missing_discharge_mask,
        "EncounterID"
    ]
    .map(encounter_end_lookup)
)

In [465]:
print(
    "Missing discharge after recovery:",
    admissions_clean["DischargeDateTime"].isna().sum()
)

print(
    "Discharge before admission:",
    (
        admissions_clean["DischargeDateTime"]
        <
        admissions_clean["AdmissionDateTime"]
    ).sum()
)

Missing discharge after recovery: 0
Discharge before admission: 0


In [466]:
admissions_clean["LengthOfStayDays"] = (
    admissions_clean["DischargeDateTime"]
    -
    admissions_clean["AdmissionDateTime"]
).dt.total_seconds() / 86400

In [467]:
print(
    "Missing LOS:",
    admissions_clean["LengthOfStayDays"].isna().sum()
)

print(
    "Negative LOS:",
    (
        admissions_clean["LengthOfStayDays"] < 0
    ).sum()
)

print(
    "Minimum LOS:",
    admissions_clean["LengthOfStayDays"].min()
)

print(
    "Maximum LOS:",
    admissions_clean["LengthOfStayDays"].max()
)

print("\nLOS percentiles:")

print(
    admissions_clean["LengthOfStayDays"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
    .round(2)
)

Missing LOS: 0
Negative LOS: 0
Minimum LOS: 0.02020833333333333
Maximum LOS: 1224.0672222222222

LOS percentiles:
0.50       3.98
0.75       5.50
0.90       6.42
0.95       6.74
0.99       6.98
1.00    1224.07
Name: LengthOfStayDays, dtype: float64


In [468]:
admission_discharge_check = admissions_clean[
    [
        "AdmissionID",
        "EncounterID",
        "AdmissionDateTime",
        "DischargeDateTime",
        "LengthOfStayDays"
    ]
].copy()

# Correct encounter-end timestamp from the fixed encounter table
admission_discharge_check[
    "CorrectedEncounterEndDateTime"
] = admission_discharge_check[
    "EncounterID"
].map(encounter_end_lookup)

# What LOS would be if we used the corrected encounter end?
admission_discharge_check[
    "LOSUsingCorrectedEncounterEnd"
] = (
    admission_discharge_check[
        "CorrectedEncounterEndDateTime"
    ]
    -
    admission_discharge_check[
        "AdmissionDateTime"
    ]
).dt.total_seconds() / 86400

extreme_check = admission_discharge_check[
    admission_discharge_check["LengthOfStayDays"] > 10
].copy()

print(
    "Admissions currently above 10 days:",
    len(extreme_check)
)

print(
    "Of those, corrected encounter end would make LOS <= 10 days:",
    (
        extreme_check["LOSUsingCorrectedEncounterEnd"] <= 10
    ).sum()
)

extreme_check[
    [
        "AdmissionID",
        "EncounterID",
        "AdmissionDateTime",
        "DischargeDateTime",
        "CorrectedEncounterEndDateTime",
        "LengthOfStayDays",
        "LOSUsingCorrectedEncounterEnd"
    ]
].head(20)

Admissions currently above 10 days: 23
Of those, corrected encounter end would make LOS <= 10 days: 23


,AdmissionID,EncounterID,AdmissionDateTime,DischargeDateTime,CorrectedEncounterEndDateTime,LengthOfStayDays,LOSUsingCorrectedEncounterEnd
78,ADM000079,ENC000416,2025-12-27 21:44:00,2026-08-28 12:40:06,2025-12-28 10:24:06,243.622292,0.527847
2219,ADM002220,ENC011655,2025-12-25 15:54:00,2026-08-28 11:52:06,2025-12-26 03:46:06,245.832014,0.494514
2801,ADM002802,ENC014464,2024-12-26 08:13:00,2026-08-28 08:50:48,2024-12-26 17:03:48,610.026250,0.368611
2914,ADM002915,ENC015053,2025-12-03 16:10:00,2026-08-28 21:05:06,2025-12-04 13:15:06,268.204931,0.878542
2933,ADM002934,ENC015159,2025-10-26 18:37:00,2026-08-28 02:20:48,2025-10-26 20:57:48,305.322083,0.097778
3091,ADM003092,ENC016012,2024-10-21 18:07:00,2026-08-28 02:35:30,2024-10-21 20:42:30,675.353125,0.107986
3686,ADM003687,ENC019160,2025-11-05 20:33:00,2026-08-28 07:51:36,2025-11-06 04:24:36,295.471250,0.327500
6507,ADM006508,ENC033213,2025-11-06 06:30:00,2026-08-28 20:21:54,2025-11-07 02:51:54,295.577708,0.848542
6814,ADM006815,ENC034641,2025-11-10 07:27:00,2026-08-28 12:59:00,2025-11-10 20:26:00,291.230556,0.540972
6905,ADM006906,ENC035116,2024-11-03 07:11:00,2026-08-28 00:57:00,2024-11-03 08:08:00,662.740278,0.039583


In [194]:
missing_discharge_after = (
    admissions_clean[
        "DischargeDateTime"
    ].isna().sum()
)

recovered_discharge_count = (
    missing_discharge_before
    -
    missing_discharge_after
)

print(
    "Missing discharge after recovery:",
    missing_discharge_after
)

print(
    "Discharge dates recovered:",
    recovered_discharge_count
)

Missing discharge after recovery: 74
Discharge dates recovered: 50


In [195]:
remaining_missing_discharge = (
    admissions_clean[
        admissions_clean["DischargeDateTime"].isna()
    ][
        [
            "AdmissionID",
            "EncounterID",
            "PatientID",
            "AdmissionDateTime"
        ]
    ]
    .copy()
)

remaining_missing_discharge[
    "EncounterExists"
] = remaining_missing_discharge[
    "EncounterID"
].isin(
    encounters_clean["EncounterID"]
)

remaining_missing_discharge[
    "LinkedEncounterEnd"
] = remaining_missing_discharge[
    "EncounterID"
].map(
    encounter_end_lookup
)

print(
    "Remaining missing discharge:",
    len(remaining_missing_discharge)
)

print(
    "EncounterID not found:",
    (
        ~remaining_missing_discharge[
            "EncounterExists"
        ]
    ).sum()
)

print(
    "Encounter exists but EncounterEndDateTime missing:",
    (
        remaining_missing_discharge[
            "EncounterExists"
        ]
        &
        remaining_missing_discharge[
            "LinkedEncounterEnd"
        ].isna()
    ).sum()
)

remaining_missing_discharge.head(20)

Remaining missing discharge: 74
EncounterID not found: 0
Encounter exists but EncounterEndDateTime missing: 74


,AdmissionID,EncounterID,PatientID,AdmissionDateTime,EncounterExists,LinkedEncounterEnd
396,ADM000397,ENC002085,PAT006156,2024-08-25 14:15:00,True,NaT
529,ADM000530,ENC002811,PAT001775,2024-12-15 09:25:00,True,NaT
563,ADM000564,ENC003037,PAT000347,2024-07-20 10:33:00,True,NaT
817,ADM000818,ENC004313,PAT004991,2025-10-12 09:36:00,True,NaT
1087,ADM001088,ENC005766,PAT005445,2024-11-27 19:54:00,True,NaT
1121,ADM001122,ENC005940,PAT002136,2025-10-24 09:52:00,True,NaT
1160,ADM001161,ENC006186,PAT001620,2025-11-25 22:21:00,True,NaT
1187,ADM001188,ENC006324,PAT002704,2025-11-29 09:59:00,True,NaT
1388,ADM001389,ENC007428,PAT004147,2025-12-08 19:45:00,True,NaT
1473,ADM001474,ENC007844,PAT001437,2025-10-06 22:10:00,True,NaT


In [196]:
print(
    "Admission EncounterID examples:"
)

print(
    remaining_missing_discharge[
        "EncounterID"
    ].head(10).tolist()
)

print(
    "\nMatching rows from encounters:"
)

print(
    encounters_clean[
        encounters_clean[
            "EncounterID"
        ].isin(
            remaining_missing_discharge[
                "EncounterID"
            ]
        )
    ][
        [
            "EncounterID",
            "EncounterType",
            "ArrivalDateTime",
            "EncounterEndDateTime"
        ]
    ]
    .head(20)
)

Admission EncounterID examples:
['ENC002085', 'ENC002811', 'ENC003037', 'ENC004313', 'ENC005766', 'ENC005940', 'ENC006186', 'ENC006324', 'ENC007428', 'ENC007844']

Matching rows from encounters:
      EncounterID EncounterType     ArrivalDateTime EncounterEndDateTime
2084    ENC002085     Inpatient 2024-08-25 14:15:00                  NaT
2810    ENC002811     Inpatient 2024-12-15 09:25:00                  NaT
3036    ENC003037     Inpatient 2024-07-20 10:33:00                  NaT
4312    ENC004313     Inpatient 2025-10-12 09:36:00                  NaT
5765    ENC005766     Inpatient 2024-11-27 19:54:00                  NaT
5939    ENC005940     Inpatient 2025-10-24 09:52:00                  NaT
6185    ENC006186     Inpatient 2025-11-25 22:21:00                  NaT
6323    ENC006324     Inpatient 2025-11-29 09:59:00                  NaT
7427    ENC007428     Inpatient 2025-12-08 19:45:00                  NaT
7843    ENC007844     Inpatient 2025-10-06 22:10:00                  NaT
78

In [445]:
affected_encounter_ids = (
    remaining_missing_discharge[
        "EncounterID"
    ].unique()
)

raw_end_check = (
    encounters_raw[
        encounters_raw[
            "EncounterID"
        ].isin(
            affected_encounter_ids
        )
    ][
        [
            "EncounterID",
            "EncounterType",
            "EncounterStatus",
            "ArrivalDateTime",
            "EncounterEndDateTime"
        ]
    ]
    .copy()
)

print(
    "Affected encounters:",
    len(raw_end_check)
)

print(
    "Raw EncounterEndDateTime missing:",
    raw_end_check[
        "EncounterEndDateTime"
    ].isna().sum()
)

raw_end_check.head(20)

Affected encounters: 74
Raw EncounterEndDateTime missing: 0


,EncounterID,EncounterType,EncounterStatus,ArrivalDateTime,EncounterEndDateTime
2084,ENC002085,Inpatient,Completed,25-08-2024 14:15,43:18.4
2810,ENC002811,Inpatient,Completed,15-12-2024 09:25,32:43.5
3036,ENC003037,Inpatient,Completed,20-07-2024 10:33,33:58.6
4312,ENC004313,Inpatient,Completed,12-10-2025 09:36,38:39.8
5765,ENC005766,Inpatient,Completed,27-11-2024 19:54,37:28.1
5939,ENC005940,Inpatient,Completed,24-10-2025 09:52,45:23.6
6185,ENC006186,Inpatient,Completed,25-11-2025 22:21,35:40.8
6323,ENC006324,Inpatient,Completed,29-11-2025 09:59,33:15.2
7427,ENC007428,Inpatient,Completed,08-12-2025 19:45,46:59.3
7843,ENC007844,Inpatient,Completed,06-10-2025 22:10,44:53.4


In [198]:
raw_clean_end_check = (
    encounters_raw[
        encounters_raw[
            "EncounterID"
        ].isin(
            affected_encounter_ids
        )
    ][
        [
            "EncounterID",
            "EncounterEndDateTime"
        ]
    ]
    .merge(
        encounters_clean[
            [
                "EncounterID",
                "EncounterEndDateTime"
            ]
        ],
        on="EncounterID",
        how="left",
        suffixes=(
            "_Raw",
            "_Clean"
        )
    )
)

raw_clean_end_check.head(20)

,EncounterID,EncounterEndDateTime_Raw,EncounterEndDateTime_Clean
0,ENC002085,43:18.4,NaT
1,ENC002811,32:43.5,NaT
2,ENC003037,33:58.6,NaT
3,ENC004313,38:39.8,NaT
4,ENC005766,37:28.1,NaT
5,ENC005940,45:23.6,NaT
6,ENC006186,35:40.8,NaT
7,ENC006324,33:15.2,NaT
8,ENC007428,46:59.3,NaT
9,ENC007844,44:53.4,NaT


In [199]:
print(
    "Raw has value but Clean is missing:",
    (
        raw_clean_end_check[
            "EncounterEndDateTime_Raw"
        ].notna()
        &
        raw_clean_end_check[
            "EncounterEndDateTime_Clean"
        ].isna()
    ).sum()
)

print(
    "Missing in both Raw and Clean:",
    (
        raw_clean_end_check[
            "EncounterEndDateTime_Raw"
        ].isna()
        &
        raw_clean_end_check[
            "EncounterEndDateTime_Clean"
        ].isna()
    ).sum()
)

Raw has value but Clean is missing: 74
Missing in both Raw and Clean: 0


In [200]:
raw_encounter_end_lookup = (
    encounters_raw
    .assign(
        EncounterEndDateTimeParsed=pd.to_datetime(
            encounters_raw["EncounterEndDateTime"],
            format="mixed",
            errors="coerce"
        )
    )
    .set_index("EncounterID")[
        "EncounterEndDateTimeParsed"
    ]
)

In [201]:
lost_encounter_end_mask = (
    encounters_clean[
        "EncounterEndDateTime"
    ].isna()
    &
    encounters_clean[
        "EncounterID"
    ].isin(
        affected_encounter_ids
    )
)

print(
    "Encounter end timestamps to restore:",
    lost_encounter_end_mask.sum()
)

Encounter end timestamps to restore: 74


In [202]:
encounters_clean.loc[
    lost_encounter_end_mask,
    "EncounterEndDateTime"
] = (
    encounters_clean.loc[
        lost_encounter_end_mask,
        "EncounterID"
    ]
    .map(
        raw_encounter_end_lookup
    )
)

In [203]:
print(
    "Still missing EncounterEndDateTime:",
    encounters_clean.loc[
        encounters_clean[
            "EncounterID"
        ].isin(affected_encounter_ids),
        "EncounterEndDateTime"
    ].isna().sum()
)

Still missing EncounterEndDateTime: 74


In [204]:
affected_raw_end = (
    encounters_raw[
        encounters_raw["EncounterID"].isin(
            affected_encounter_ids
        )
    ][
        [
            "EncounterID",
            "EncounterEndDateTime"
        ]
    ]
    .copy()
)

affected_raw_end.head(20)

,EncounterID,EncounterEndDateTime
2084,ENC002085,43:18.4
2810,ENC002811,32:43.5
3036,ENC003037,33:58.6
4312,ENC004313,38:39.8
5765,ENC005766,37:28.1
5939,ENC005940,45:23.6
6185,ENC006186,35:40.8
6323,ENC006324,33:15.2
7427,ENC007428,46:59.3
7843,ENC007844,44:53.4


In [205]:
for value in affected_raw_end[
    "EncounterEndDateTime"
].head(10):

    print(repr(value))

'43:18.4'
'32:43.5'
'33:58.6'
'38:39.8'
'37:28.1'
'45:23.6'
'35:40.8'
'33:15.2'
'46:59.3'
'44:53.4'


In [206]:
def parse_timestamp_flexible(value):

    if pd.isna(value):
        return pd.NaT

    try:
        return pd.Timestamp(
            str(value).strip()
        )

    except Exception:
        return pd.NaT

In [207]:
affected_raw_end[
    "EncounterEndDateTimeParsed"
] = affected_raw_end[
    "EncounterEndDateTime"
].apply(
    parse_timestamp_flexible
)

In [208]:
print(
    "Affected raw rows:",
    len(affected_raw_end)
)

print(
    "Still failing to parse:",
    affected_raw_end[
        "EncounterEndDateTimeParsed"
    ].isna().sum()
)

Affected raw rows: 74
Still failing to parse: 74


In [209]:
print(
    affected_raw_end[
        [
            "EncounterID",
            "EncounterEndDateTime"
        ]
    ].head(20).to_string(index=False)
)

EncounterID EncounterEndDateTime
  ENC002085              43:18.4
  ENC002811              32:43.5
  ENC003037              33:58.6
  ENC004313              38:39.8
  ENC005766              37:28.1
  ENC005940              45:23.6
  ENC006186              35:40.8
  ENC006324              33:15.2
  ENC007428              46:59.3
  ENC007844              44:53.4
  ENC007865              50:49.8
  ENC009235              41:09.5
  ENC010221              37:22.5
  ENC010546              31:37.3
  ENC012003              49:53.1
  ENC013364              42:14.1
  ENC013683              40:33.4
  ENC014686              56:46.3
  ENC015520              37:09.5
  ENC018045              39:50.5


In [210]:
print("\nExact raw representations:")

for value in affected_raw_end[
    "EncounterEndDateTime"
].head(20):
    print(repr(value), type(value))


Exact raw representations:
'43:18.4' <class 'str'>
'32:43.5' <class 'str'>
'33:58.6' <class 'str'>
'38:39.8' <class 'str'>
'37:28.1' <class 'str'>
'45:23.6' <class 'str'>
'35:40.8' <class 'str'>
'33:15.2' <class 'str'>
'46:59.3' <class 'str'>
'44:53.4' <class 'str'>
'50:49.8' <class 'str'>
'41:09.5' <class 'str'>
'37:22.5' <class 'str'>
'31:37.3' <class 'str'>
'49:53.1' <class 'str'>
'42:14.1' <class 'str'>
'40:33.4' <class 'str'>
'56:46.3' <class 'str'>
'37:09.5' <class 'str'>
'39:50.5' <class 'str'>


In [211]:
raw_end_text = (
    affected_raw_end[
        "EncounterEndDateTime"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

placeholder_values = [
    "",
    "nat",
    "nan",
    "none",
    "null"
]

print(
    "Placeholder / semantic missing values:",
    raw_end_text.isin(
        placeholder_values
    ).sum()
)

print("\nRaw value frequencies:")

print(
    raw_end_text.value_counts(
        dropna=False
    ).head(20)
)

Placeholder / semantic missing values: 0

Raw value frequencies:
EncounterEndDateTime
43:18.4    1
32:43.5    1
33:58.6    1
38:39.8    1
37:28.1    1
45:23.6    1
35:40.8    1
33:15.2    1
46:59.3    1
44:53.4    1
50:49.8    1
41:09.5    1
37:22.5    1
31:37.3    1
49:53.1    1
42:14.1    1
40:33.4    1
56:46.3    1
37:09.5    1
39:50.5    1
Name: count, dtype: int64


In [212]:
malformed_end_check = (
    encounters_raw[
        encounters_raw["EncounterID"].isin(
            affected_encounter_ids
        )
    ][
        [
            "EncounterID",
            "EncounterType",
            "ArrivalDateTime",
            "EncounterEndDateTime"
        ]
    ]
    .copy()
)

duration_parts = (
    malformed_end_check[
        "EncounterEndDateTime"
    ]
    .astype(str)
    .str.strip()
    .str.extract(
        r"^(?P<Hours>\d+):(?P<Minutes>\d{2})\.(?P<Fraction>\d+)$"
    )
)

malformed_end_check = pd.concat(
    [
        malformed_end_check.reset_index(drop=True),
        duration_parts.reset_index(drop=True)
    ],
    axis=1
)

malformed_end_check[
    "Hours"
] = pd.to_numeric(
    malformed_end_check["Hours"],
    errors="coerce"
)

malformed_end_check[
    "Minutes"
] = pd.to_numeric(
    malformed_end_check["Minutes"],
    errors="coerce"
)

In [213]:
print(
    "Rows matching duration-like pattern:",
    malformed_end_check[
        "Hours"
    ].notna().sum()
)

print(
    "\nEncounter types:"
)

print(
    malformed_end_check[
        "EncounterType"
    ].value_counts()
)

print(
    "\nHours summary:"
)

print(
    malformed_end_check[
        "Hours"
    ].describe()
)

print(
    "\nMinutes range:"
)

print(
    malformed_end_check[
        "Minutes"
    ].min(),
    "to",
    malformed_end_check[
        "Minutes"
    ].max()
)

Rows matching duration-like pattern: 74

Encounter types:
EncounterType
Inpatient    74
Name: count, dtype: int64

Hours summary:
count    74.000000
mean     40.027027
std       9.061393
min      24.000000
25%      33.000000
50%      38.000000
75%      47.000000
max      58.000000
Name: Hours, dtype: float64

Minutes range:
0 to 59


In [214]:
malformed_end_check[
    "FractionValue"
] = pd.to_numeric(
    "0." + malformed_end_check["Fraction"].astype(str),
    errors="coerce"
).fillna(0)

In [215]:
malformed_end_check[
    "RecoveredDurationMinutes"
] = (
    malformed_end_check["Hours"] * 60
    +
    malformed_end_check["Minutes"]
    +
    malformed_end_check["FractionValue"]
)

In [216]:
malformed_end_check[
    "RecoveredDurationMinutes"
].describe()

count      74.000000
mean     2432.216216
std       547.774398
min      1480.800000
25%      2026.750000
50%      2310.350000
75%      2864.175000
max      3530.000000
Name: RecoveredDurationMinutes, dtype: float64

In [217]:
malformed_end_check[
    "ArrivalDateTimeParsed"
] = pd.to_datetime(
    malformed_end_check[
        "ArrivalDateTime"
    ],
    format="mixed",
    errors="coerce"
)

In [218]:
print(
    "Missing ArrivalDateTime after parsing:",
    malformed_end_check[
        "ArrivalDateTimeParsed"
    ].isna().sum()
)

Missing ArrivalDateTime after parsing: 0


In [219]:
malformed_end_check[
    "RecoveredEncounterEndDateTime"
] = (
    malformed_end_check[
        "ArrivalDateTimeParsed"
    ]
    +
    pd.to_timedelta(
        malformed_end_check[
            "RecoveredDurationMinutes"
        ],
        unit="m"
    )
)

In [220]:
malformed_end_check[
    [
        "EncounterID",
        "ArrivalDateTime",
        "EncounterEndDateTime",
        "RecoveredDurationMinutes",
        "RecoveredEncounterEndDateTime"
    ]
].head(20)

,EncounterID,ArrivalDateTime,EncounterEndDateTime,RecoveredDurationMinutes,RecoveredEncounterEndDateTime
0,ENC002085,25-08-2024 14:15,43:18.4,2598.4,2024-08-27 09:33:24
1,ENC002811,15-12-2024 09:25,32:43.5,1963.5,2024-12-16 18:08:30
2,ENC003037,20-07-2024 10:33,33:58.6,2038.6,2024-07-21 20:31:36
3,ENC004313,12-10-2025 09:36,38:39.8,2319.8,2025-12-12 00:15:48
4,ENC005766,27-11-2024 19:54,37:28.1,2248.1,2024-11-29 09:22:06
5,ENC005940,24-10-2025 09:52,45:23.6,2723.6,2025-10-26 07:15:36
6,ENC006186,25-11-2025 22:21,35:40.8,2140.8,2025-11-27 10:01:48
7,ENC006324,29-11-2025 09:59,33:15.2,1995.2,2025-11-30 19:14:12
8,ENC007428,08-12-2025 19:45,46:59.3,2819.3,2025-08-14 18:44:18
9,ENC007844,06-10-2025 22:10,44:53.4,2693.4,2025-06-12 19:03:24


In [221]:
malformed_end_check[
    "RecoveredDurationHours"
] = (
    malformed_end_check[
        "RecoveredEncounterEndDateTime"
    ]
    -
    malformed_end_check[
        "ArrivalDateTimeParsed"
    ]
).dt.total_seconds() / 3600

In [222]:
malformed_end_check[
    "RecoveredDurationHours"
].describe()

count    74.000000
mean     40.536937
std       9.129573
min      24.680000
25%      33.779167
50%      38.505833
75%      47.736250
max      58.833333
Name: RecoveredDurationHours, dtype: float64

In [446]:
recovered_end_lookup = (
    malformed_end_check
    .set_index("EncounterID")[
        "RecoveredEncounterEndDateTime"
    ]
)

recover_malformed_end_mask = (
    encounters_clean["EncounterID"].isin(
        recovered_end_lookup.index
    )
    &
    encounters_clean[
        "EncounterEndDateTime"
    ].isna()
)

print(
    "Encounter end timestamps to reconstruct:",
    recover_malformed_end_mask.sum()
)

Encounter end timestamps to reconstruct: 0


In [224]:
encounters_clean.loc[
    recover_malformed_end_mask,
    "EncounterEndDateTime"
] = (
    encounters_clean.loc[
        recover_malformed_end_mask,
        "EncounterID"
    ]
    .map(recovered_end_lookup)
)

In [451]:
encounters_clean["EncounterEndRecoveredFlag"] = 0

encounters_clean.loc[
    recover_malformed_end_mask,
    "EncounterEndRecoveredFlag"
] = 1

In [452]:
print(
    encounters_clean["EncounterEndRecoveredFlag"]
    .value_counts()
)

EncounterEndRecoveredFlag
0    90000
Name: count, dtype: int64


In [226]:
print(
    "Still missing among affected encounters:",
    encounters_clean.loc[
        encounters_clean["EncounterID"].isin(
            affected_encounter_ids
        ),
        "EncounterEndDateTime"
    ].isna().sum()
)

Still missing among affected encounters: 0


In [227]:
encounter_end_lookup = (
    encounters_clean
    .set_index("EncounterID")[
        "EncounterEndDateTime"
    ]
)

In [228]:
remaining_discharge_mask = (
    admissions_clean[
        "DischargeDateTime"
    ].isna()
)

print(
    "Missing before:",
    remaining_discharge_mask.sum()
)

admissions_clean.loc[
    remaining_discharge_mask,
    "DischargeDateTime"
] = (
    admissions_clean.loc[
        remaining_discharge_mask,
        "EncounterID"
    ]
    .map(encounter_end_lookup)
)

print(
    "Missing after:",
    admissions_clean[
        "DischargeDateTime"
    ].isna().sum()
)

Missing before: 74
Missing after: 0


In [229]:
encounters_clean[
    "EncounterDurationHours"
] = (
    encounters_clean["EncounterEndDateTime"]
    -
    encounters_clean["ArrivalDateTime"]
).dt.total_seconds() / 3600

In [230]:
encounters_clean[
    "EncounterDurationValidFlag"
] = (
    encounters_clean["ArrivalDateTime"].notna()
    &
    encounters_clean["EncounterEndDateTime"].notna()
    &
    (
        encounters_clean["EncounterEndDateTime"]
        >=
        encounters_clean["ArrivalDateTime"]
    )
).astype(int)

In [231]:
print(
    "Negative encounter durations:",
    (
        encounters_clean[
            "EncounterDurationHours"
        ] < 0
    ).sum()
)

print(
    "Invalid encounter durations:",
    (
        encounters_clean[
            "EncounterDurationValidFlag"
        ] == 0
    ).sum()
)

Negative encounter durations: 17
Invalid encounter durations: 53966


In [232]:
encounters_clean.to_csv(
    CLEANED_DATA_PATH / "encounters_clean.csv",
    index=False
)

In [233]:
discharge_before_admission_mask = (
    admissions_clean["DischargeDateTime"].notna()
    &
    admissions_clean["AdmissionDateTime"].notna()
    &
    (
        admissions_clean["DischargeDateTime"]
        <
        admissions_clean["AdmissionDateTime"]
    )
)

print(
    "Discharge before Admission:",
    discharge_before_admission_mask.sum()
)

Discharge before Admission: 70


In [234]:
candidate_encounter_end = (
    admissions_clean["EncounterID"]
    .map(encounter_end_lookup)
)

recoverable_bad_discharge_mask = (
    discharge_before_admission_mask
    &
    candidate_encounter_end.notna()
    &
    (
        candidate_encounter_end
        >=
        admissions_clean[
            "AdmissionDateTime"
        ]
    )
)

print(
    "Recoverable invalid discharge timestamps:",
    recoverable_bad_discharge_mask.sum()
)

Recoverable invalid discharge timestamps: 23


In [235]:
admissions_clean.loc[
    recoverable_bad_discharge_mask,
    "DischargeDateTime"
] = candidate_encounter_end[
    recoverable_bad_discharge_mask
]

In [236]:
log_cleaning_action(
    "5.5",
    "admissions",
    "DischargeDateTime",
    "Corrected discharge-before-admission timestamps using linked EncounterEndDateTime",
    recoverable_bad_discharge_mask.sum(),
    "Linked inpatient encounter provided a deterministic valid end timestamp"
)

In [237]:
remaining_bad_discharge_mask = (
    admissions_clean["DischargeDateTime"].notna()
    &
    admissions_clean["AdmissionDateTime"].notna()
    &
    (
        admissions_clean["DischargeDateTime"]
        <
        admissions_clean["AdmissionDateTime"]
    )
)

print(
    "Remaining discharge-before-admission:",
    remaining_bad_discharge_mask.sum()
)

Remaining discharge-before-admission: 47


In [238]:
remaining_bad_discharge_detail = (
    admissions_clean[
        remaining_bad_discharge_mask
    ][
        [
            "AdmissionID",
            "EncounterID",
            "PatientID",
            "AdmissionDateTime",
            "DischargeDateTime"
        ]
    ]
    .copy()
)

encounter_arrival_lookup = (
    encounters_clean
    .set_index("EncounterID")[
        "ArrivalDateTime"
    ]
)

encounter_end_lookup = (
    encounters_clean
    .set_index("EncounterID")[
        "EncounterEndDateTime"
    ]
)

remaining_bad_discharge_detail[
    "LinkedArrivalDateTime"
] = (
    remaining_bad_discharge_detail[
        "EncounterID"
    ].map(encounter_arrival_lookup)
)

remaining_bad_discharge_detail[
    "LinkedEncounterEnd"
] = (
    remaining_bad_discharge_detail[
        "EncounterID"
    ].map(encounter_end_lookup)
)

remaining_bad_discharge_detail.head(20)

,AdmissionID,EncounterID,PatientID,AdmissionDateTime,DischargeDateTime,LinkedArrivalDateTime,LinkedEncounterEnd
134,ADM000135,ENC000750,PAT003724,2025-06-05 13:17:00,2025-06-03 19:17:00,2025-06-05 13:17:00,NaT
949,ADM000950,ENC005000,PAT003258,2025-12-26 18:51:00,2025-12-26 04:51:00,2025-12-26 18:51:00,NaT
1284,ADM001285,ENC006818,PAT004813,2025-02-27 14:16:00,2025-02-26 07:16:00,2025-02-27 14:16:00,NaT
1388,ADM001389,ENC007428,PAT004147,2025-12-08 19:45:00,2025-08-14 18:44:18,2025-12-08 19:45:00,2025-08-14 18:44:18
1473,ADM001474,ENC007844,PAT001437,2025-10-06 22:10:00,2025-06-12 19:03:24,2025-10-06 22:10:00,2025-06-12 19:03:24
1478,ADM001479,ENC007865,PAT005164,2025-10-08 16:36:00,2025-08-12 19:25:48,2025-10-08 16:36:00,2025-08-12 19:25:48
1606,ADM001607,ENC008544,PAT001344,2023-09-04 18:03:00,2023-09-02 22:03:00,2023-09-04 18:03:00,NaT
1750,ADM001751,ENC009235,PAT001714,2025-12-03 17:51:00,2025-03-14 11:00:30,2025-12-03 17:51:00,2025-03-14 11:00:30
1957,ADM001958,ENC010221,PAT008013,2024-10-09 12:07:00,2024-09-12 01:29:30,2024-10-09 12:07:00,2024-09-12 01:29:30
2571,ADM002572,ENC013364,PAT009998,2024-10-04 22:11:00,2024-04-12 16:25:06,2024-10-04 22:11:00,2024-04-12 16:25:06


In [239]:
print(
    "Remaining bad admissions:",
    len(remaining_bad_discharge_detail)
)

print(
    "Linked Arrival missing:",
    remaining_bad_discharge_detail[
        "LinkedArrivalDateTime"
    ].isna().sum()
)

print(
    "Linked Encounter End missing:",
    remaining_bad_discharge_detail[
        "LinkedEncounterEnd"
    ].isna().sum()
)

print(
    "Linked Encounter End before Admission:",
    (
        remaining_bad_discharge_detail[
            "LinkedEncounterEnd"
        ]
        <
        remaining_bad_discharge_detail[
            "AdmissionDateTime"
        ]
    ).sum()
)

print(
    "Linked Encounter End before Linked Arrival:",
    (
        remaining_bad_discharge_detail[
            "LinkedEncounterEnd"
        ]
        <
        remaining_bad_discharge_detail[
            "LinkedArrivalDateTime"
        ]
    ).sum()
)

Remaining bad admissions: 47
Linked Arrival missing: 0
Linked Encounter End missing: 30
Linked Encounter End before Admission: 17
Linked Encounter End before Linked Arrival: 17


In [240]:
remaining_bad_discharge_detail[
    "AdmissionArrivalDifferenceMinutes"
] = (
    remaining_bad_discharge_detail[
        "AdmissionDateTime"
    ]
    -
    remaining_bad_discharge_detail[
        "LinkedArrivalDateTime"
    ]
).dt.total_seconds() / 60

print(
    "AdmissionDateTime != linked ArrivalDateTime:",
    (
        remaining_bad_discharge_detail[
            "AdmissionArrivalDifferenceMinutes"
        ].abs() > 0.01
    ).sum()
)

print(
    remaining_bad_discharge_detail[
        "AdmissionArrivalDifferenceMinutes"
    ].describe()
)

AdmissionDateTime != linked ArrivalDateTime: 0
count    47.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: AdmissionArrivalDifferenceMinutes, dtype: float64


In [242]:
problem_encounter_ids = (
    remaining_bad_discharge_detail[
        "EncounterID"
    ].unique()
)

problem_end_raw_check = (
    encounters_raw[
        encounters_raw[
            "EncounterID"
        ].isin(
            problem_encounter_ids
        )
    ][
        [
            "EncounterID",
            "EncounterType",
            "ArrivalDateTime",
            "EncounterEndDateTime"
        ]
    ]
    .copy()
)

problem_end_raw_check.head(20)

,EncounterID,EncounterType,ArrivalDateTime,EncounterEndDateTime
749,ENC000750,Inpatient,05-06-2025 13:17,34:48.1
4999,ENC005000,Inpatient,26-12-2025 18:51,47:57.3
6817,ENC006818,Inpatient,27-02-2025 14:16,25:09.7
7427,ENC007428,Inpatient,08-12-2025 19:45,46:59.3
7843,ENC007844,Inpatient,06-10-2025 22:10,44:53.4
7864,ENC007865,Inpatient,08-10-2025 16:36,50:49.8
8543,ENC008544,Inpatient,04-09-2023 18:03,52:20.2
9234,ENC009235,Inpatient,03-12-2025 17:51,41:09.5
10220,ENC010221,Inpatient,09-10-2024 12:07,37:22.5
13363,ENC013364,Inpatient,04-10-2024 22:11,42:14.1


In [243]:
problem_end_check = (
    problem_end_raw_check
    .merge(
        encounters_clean[
            [
                "EncounterID",
                "ArrivalDateTime",
                "EncounterEndDateTime"
            ]
        ],
        on="EncounterID",
        how="left",
        suffixes=(
            "_Raw",
            "_Clean"
        )
    )
)

problem_end_check.head(20)

,EncounterID,EncounterType,ArrivalDateTime_Raw,EncounterEndDateTime_Raw,ArrivalDateTime_Clean,EncounterEndDateTime_Clean
0,ENC000750,Inpatient,05-06-2025 13:17,34:48.1,2025-06-05 13:17:00,NaT
1,ENC005000,Inpatient,26-12-2025 18:51,47:57.3,2025-12-26 18:51:00,NaT
2,ENC006818,Inpatient,27-02-2025 14:16,25:09.7,2025-02-27 14:16:00,NaT
3,ENC007428,Inpatient,08-12-2025 19:45,46:59.3,2025-12-08 19:45:00,2025-08-14 18:44:18
4,ENC007844,Inpatient,06-10-2025 22:10,44:53.4,2025-10-06 22:10:00,2025-06-12 19:03:24
5,ENC007865,Inpatient,08-10-2025 16:36,50:49.8,2025-10-08 16:36:00,2025-08-12 19:25:48
6,ENC008544,Inpatient,04-09-2023 18:03,52:20.2,2023-09-04 18:03:00,NaT
7,ENC009235,Inpatient,03-12-2025 17:51,41:09.5,2025-12-03 17:51:00,2025-03-14 11:00:30
8,ENC010221,Inpatient,09-10-2024 12:07,37:22.5,2024-10-09 12:07:00,2024-09-12 01:29:30
9,ENC013364,Inpatient,04-10-2024 22:11,42:14.1,2024-10-04 22:11:00,2024-04-12 16:25:06


In [244]:
print(
    problem_end_check[
        "EncounterEndDateTime_Raw"
    ]
    .astype(str)
    .head(30)
    .to_string(index=False)
)

34:48.1
47:57.3
25:09.7
46:59.3
44:53.4
50:49.8
52:20.2
41:09.5
37:22.5
42:14.1
24:12.7
26:10.3
36:00.8
31:58.1
56:37.6
29:24.8
32:54.4
37:31.0
49:15.8
54:49.9
33:07.5
56:53.9
46:26.5
58:50.0
45:23.3
47:42.6
31:26.5
44:04.8
35:25.5
58:03.7


In [245]:
problem_raw_text = (
    problem_end_check[
        "EncounterEndDateTime_Raw"
    ]
    .astype(str)
    .str.strip()
)

duration_like_mask = (
    problem_raw_text.str.match(
        r"^\d+:\d{2}\.\d+$",
        na=False
    )
)

print(
    "Duration-like malformed values:",
    duration_like_mask.sum()
)

print(
    "Other raw formats:",
    (
        ~duration_like_mask
    ).sum()
)

Duration-like malformed values: 47
Other raw formats: 0


In [246]:
print(
    problem_end_check[
        "EncounterType"
    ].value_counts()
)

EncounterType
Inpatient    47
Name: count, dtype: int64


In [247]:
duration_parts = (
    problem_raw_text
    .str.extract(
        r"^(?P<Hours>\d+):(?P<Minutes>\d{2})\.(?P<Fraction>\d+)$"
    )
)

problem_end_check[
    "DurationHours"
] = pd.to_numeric(
    duration_parts["Hours"],
    errors="coerce"
)

problem_end_check[
    "DurationMinutes"
] = pd.to_numeric(
    duration_parts["Minutes"],
    errors="coerce"
)

problem_end_check[
    "DurationFraction"
] = pd.to_numeric(
    "0." + duration_parts[
        "Fraction"
    ].astype(str),
    errors="coerce"
)

In [248]:
print(
    problem_end_check[
        "DurationHours"
    ].describe()
)

print(
    "Minute range:",
    problem_end_check[
        "DurationMinutes"
    ].min(),
    "to",
    problem_end_check[
        "DurationMinutes"
    ].max()
)

print(
    "Rows successfully interpreted as duration:",
    problem_end_check[
        "DurationHours"
    ].notna().sum()
)

count    47.000000
mean     40.425532
std      10.042831
min      24.000000
25%      32.500000
50%      40.000000
75%      47.500000
max      58.000000
Name: DurationHours, dtype: float64
Minute range: 0 to 59
Rows successfully interpreted as duration: 47


In [249]:
problem_end_check[
    "EncounterType"
].value_counts()

EncounterType
Inpatient    47
Name: count, dtype: int64

In [250]:
problem_end_check[
    "RecoveredDurationMinutes"
] = (
    problem_end_check["DurationHours"] * 60
    +
    problem_end_check["DurationMinutes"]
    +
    problem_end_check[
        "DurationFraction"
    ].fillna(0)
)

In [251]:
arrival_lookup = (
    encounters_clean
    .set_index("EncounterID")[
        "ArrivalDateTime"
    ]
)

In [252]:
problem_end_check[
    "CleanArrivalDateTime"
] = (
    problem_end_check[
        "EncounterID"
    ].map(
        arrival_lookup
    )
)

In [253]:
print(
    "Missing clean ArrivalDateTime:",
    problem_end_check[
        "CleanArrivalDateTime"
    ].isna().sum()
)

Missing clean ArrivalDateTime: 0


In [254]:
problem_end_check[
    "RecoveredEncounterEndDateTime"
] = (
    problem_end_check[
        "CleanArrivalDateTime"
    ]
    +
    pd.to_timedelta(
        problem_end_check[
            "RecoveredDurationMinutes"
        ],
        unit="m"
    )
)

In [255]:
problem_end_check[
    [
        "EncounterID",
        "CleanArrivalDateTime",
        "EncounterEndDateTime_Raw",
        "RecoveredDurationMinutes",
        "RecoveredEncounterEndDateTime"
    ]
].head(20)

,EncounterID,CleanArrivalDateTime,EncounterEndDateTime_Raw,RecoveredDurationMinutes,RecoveredEncounterEndDateTime
0,ENC000750,2025-06-05 13:17:00,34:48.1,2088.1,2025-06-07 00:05:06
1,ENC005000,2025-12-26 18:51:00,47:57.3,2877.3,2025-12-28 18:48:18
2,ENC006818,2025-02-27 14:16:00,25:09.7,1509.7,2025-02-28 15:25:42
3,ENC007428,2025-12-08 19:45:00,46:59.3,2819.3,2025-12-10 18:44:18
4,ENC007844,2025-10-06 22:10:00,44:53.4,2693.4,2025-10-08 19:03:24
5,ENC007865,2025-10-08 16:36:00,50:49.8,3049.8,2025-10-10 19:25:48
6,ENC008544,2023-09-04 18:03:00,52:20.2,3140.2,2023-09-06 22:23:12
7,ENC009235,2025-12-03 17:51:00,41:09.5,2469.5,2025-12-05 11:00:30
8,ENC010221,2024-10-09 12:07:00,37:22.5,2242.5,2024-10-11 01:29:30
9,ENC013364,2024-10-04 22:11:00,42:14.1,2534.1,2024-10-06 16:25:06


In [256]:
problem_end_check[
    "RecoveredDurationHoursCheck"
] = (
    problem_end_check[
        "RecoveredEncounterEndDateTime"
    ]
    -
    problem_end_check[
        "CleanArrivalDateTime"
    ]
).dt.total_seconds() / 3600

In [257]:
print(
    "Recovered End before Arrival:",
    (
        problem_end_check[
            "RecoveredEncounterEndDateTime"
        ]
        <
        problem_end_check[
            "CleanArrivalDateTime"
        ]
    ).sum()
)

print(
    problem_end_check[
        "RecoveredDurationHoursCheck"
    ].describe()
)

Recovered End before Arrival: 0
count    47.000000
mean     40.971135
std      10.077893
min      24.211667
25%      33.052500
50%      40.138333
75%      48.449167
max      58.833333
Name: RecoveredDurationHoursCheck, dtype: float64


In [258]:
problem_recovered_end_lookup = (
    problem_end_check
    .set_index("EncounterID")[
        "RecoveredEncounterEndDateTime"
    ]
)

In [259]:
problem_end_recovery_mask = (
    encounters_clean[
        "EncounterID"
    ].isin(
        problem_recovered_end_lookup.index
    )
)

print(
    "Encounter rows to repair:",
    problem_end_recovery_mask.sum()
)

Encounter rows to repair: 47


In [260]:
encounters_clean.loc[
    problem_end_recovery_mask,
    "EncounterEndDateTime"
] = (
    encounters_clean.loc[
        problem_end_recovery_mask,
        "EncounterID"
    ]
    .map(
        problem_recovered_end_lookup
    )
)

In [261]:
if "EncounterEndRecoveredFlag" not in encounters_clean.columns:
    encounters_clean[
        "EncounterEndRecoveredFlag"
    ] = 0

In [262]:
encounters_clean.loc[
    problem_end_recovery_mask,
    "EncounterEndRecoveredFlag"
] = 1

In [263]:
if "EncounterEndRecoveryMethod" not in encounters_clean.columns:
    encounters_clean[
        "EncounterEndRecoveryMethod"
    ] = pd.NA

In [264]:
encounters_clean.loc[
    problem_end_recovery_mask,
    "EncounterEndRecoveryMethod"
] = (
    "Reconstructed from ArrivalDateTime + elapsed-duration source"
)

In [265]:
print(
    "Total reconstructed encounter ends:",
    encounters_clean[
        "EncounterEndRecoveredFlag"
    ].sum()
)

Total reconstructed encounter ends: 104


In [266]:
all_reconstructed_encounter_ids = set(
    affected_encounter_ids
).union(
    set(problem_encounter_ids)
)

print(
    "Expected unique reconstructed EncounterIDs:",
    len(all_reconstructed_encounter_ids)
)

Expected unique reconstructed EncounterIDs: 104


In [267]:
print(
    "Currently flagged reconstructed encounters:",
    encounters_clean[
        "EncounterEndRecoveredFlag"
    ].sum()
)

Currently flagged reconstructed encounters: 104


In [268]:
missing_recovery_flag = encounters_clean[
    encounters_clean[
        "EncounterID"
    ].isin(
        all_reconstructed_encounter_ids
    )
    &
    (
        encounters_clean[
            "EncounterEndRecoveredFlag"
        ] != 1
    )
][
    [
        "EncounterID",
        "EncounterType",
        "ArrivalDateTime",
        "EncounterEndDateTime",
        "EncounterEndRecoveredFlag"
    ]
]

print(
    "Reconstructed encounters missing recovery flag:",
    len(missing_recovery_flag)
)

missing_recovery_flag.head(20)

Reconstructed encounters missing recovery flag: 0


,EncounterID,EncounterType,ArrivalDateTime,EncounterEndDateTime,EncounterEndRecoveredFlag


In [269]:
first_set = set(affected_encounter_ids)
second_set = set(problem_encounter_ids)

print("First batch:", len(first_set))
print("Second batch:", len(second_set))

print(
    "Overlap between batches:",
    len(first_set.intersection(second_set))
)

print(
    "Unique reconstructed encounters:",
    len(first_set.union(second_set))
)

First batch: 74
Second batch: 47
Overlap between batches: 17
Unique reconstructed encounters: 104


In [270]:
reconstructed_ids = first_set.union(
    second_set
)

reconstructed_mask = (
    encounters_clean[
        "EncounterID"
    ].isin(
        reconstructed_ids
    )
)

print(
    "Unique reconstructed encounters:",
    reconstructed_mask.sum()
)

print(
    "Missing reconstructed EncounterEndDateTime:",
    encounters_clean.loc[
        reconstructed_mask,
        "EncounterEndDateTime"
    ].isna().sum()
)

print(
    "Reconstructed End before Arrival:",
    (
        encounters_clean.loc[
            reconstructed_mask,
            "EncounterEndDateTime"
        ]
        <
        encounters_clean.loc[
            reconstructed_mask,
            "ArrivalDateTime"
        ]
    ).sum()
)

print(
    "Recovery flags:",
    encounters_clean.loc[
        reconstructed_mask,
        "EncounterEndRecoveredFlag"
    ].sum()
)

Unique reconstructed encounters: 104
Missing reconstructed EncounterEndDateTime: 0
Reconstructed End before Arrival: 0
Recovery flags: 104


In [271]:
encounter_end_lookup = (
    encounters_clean
    .set_index("EncounterID")[
        "EncounterEndDateTime"
    ]
)

In [272]:
remaining_bad_discharge_mask = (
    admissions_clean[
        "DischargeDateTime"
    ].notna()
    &
    admissions_clean[
        "AdmissionDateTime"
    ].notna()
    &
    (
        admissions_clean[
            "DischargeDateTime"
        ]
        <
        admissions_clean[
            "AdmissionDateTime"
        ]
    )
)

print(
    "Bad admission discharges before repair:",
    remaining_bad_discharge_mask.sum()
)

Bad admission discharges before repair: 47


In [273]:
corrected_admission_discharge = (
    admissions_clean[
        "EncounterID"
    ].map(
        encounter_end_lookup
    )
)

In [274]:
safe_discharge_repair_mask = (
    remaining_bad_discharge_mask
    &
    corrected_admission_discharge.notna()
    &
    (
        corrected_admission_discharge
        >=
        admissions_clean[
            "AdmissionDateTime"
        ]
    )
)

print(
    "Safely repairable:",
    safe_discharge_repair_mask.sum()
)

Safely repairable: 47


In [275]:
admissions_clean.loc[
    safe_discharge_repair_mask,
    "DischargeDateTime"
] = (
    corrected_admission_discharge[
        safe_discharge_repair_mask
    ]
)

In [276]:
print(
    "FINAL missing discharge:",
    admissions_clean[
        "DischargeDateTime"
    ].isna().sum()
)

print(
    "FINAL discharge before admission:",
    (
        admissions_clean[
            "DischargeDateTime"
        ]
        <
        admissions_clean[
            "AdmissionDateTime"
        ]
    ).sum()
)

FINAL missing discharge: 0
FINAL discharge before admission: 0


In [277]:
admissions_clean[
    "LOSValidFlag"
] = (
    admissions_clean["AdmissionDateTime"].notna()
    &
    admissions_clean["DischargeDateTime"].notna()
    &
    (
        admissions_clean["DischargeDateTime"]
        >=
        admissions_clean["AdmissionDateTime"]
    )
).astype(int)

In [278]:
admissions_clean[
    "LengthOfStayDays"
] = (
    admissions_clean["DischargeDateTime"]
    -
    admissions_clean["AdmissionDateTime"]
).dt.total_seconds() / 86400

In [279]:
admissions_clean[
    "LengthOfStayDays"
] = admissions_clean[
    "LengthOfStayDays"
].where(
    admissions_clean["LOSValidFlag"] == 1,
    np.nan
)

In [280]:
print(
    "Negative LOS:",
    (
        admissions_clean["LengthOfStayDays"] < 0
    ).sum()
)

print(
    admissions_clean[
        "LengthOfStayDays"
    ].describe()
)

Negative LOS: 0
count    17740.000000
mean         6.042963
std         35.166265
min          1.007055
25%          2.472580
50%          3.999157
75%          5.517712
max       1224.067222
Name: LengthOfStayDays, dtype: float64


In [281]:
valid_admission_types = [
    "Emergency",
    "Elective",
    "Urgent"
]

invalid_admission_type_mask = (
    ~admissions_clean[
        "AdmissionType"
    ].isin(valid_admission_types)
)

print(
    "Invalid AdmissionType:",
    invalid_admission_type_mask.sum()
)

Invalid AdmissionType: 0


In [282]:
valid_discharge_dispositions = [
    "Home",
    "Home with Services",
    "Skilled Nursing Facility",
    "Rehabilitation Facility",
    "Transfer to Another Facility",
    "Left Against Medical Advice",
    "Expired",
    "Other"
]

invalid_disposition_mask = (
    ~admissions_clean[
        "DischargeDisposition"
    ].isin(valid_discharge_dispositions)
)

print(
    "Invalid DischargeDisposition:",
    invalid_disposition_mask.sum()
)

Invalid DischargeDisposition: 0


In [283]:
print(
    "Invalid FollowUpRequiredFlag:",
    (
        admissions_clean["FollowUpRequiredFlag"].notna()
        &
        ~admissions_clean[
            "FollowUpRequiredFlag"
        ].isin([0, 1])
    ).sum()
)

print(
    "Invalid FollowUpCompletedFlag:",
    (
        admissions_clean["FollowUpCompletedFlag"].notna()
        &
        ~admissions_clean[
            "FollowUpCompletedFlag"
        ].isin([0, 1])
    ).sum()
)

Invalid FollowUpRequiredFlag: 0
Invalid FollowUpCompletedFlag: 0


In [284]:
completed_missing_followup_date_mask = (
    (
        admissions_clean[
            "FollowUpCompletedFlag"
        ] == 1
    )
    &
    admissions_clean[
        "FollowUpDate"
    ].isna()
)

print(
    "Completed follow-up missing FollowUpDate:",
    completed_missing_followup_date_mask.sum()
)

Completed follow-up missing FollowUpDate: 88


In [285]:
admissions_clean[
    "FollowUpDateValidFlag"
] = 1

admissions_clean.loc[
    completed_missing_followup_date_mask,
    "FollowUpDateValidFlag"
] = 0

In [286]:
log_cleaning_action(
    "5.5",
    "admissions",
    "FollowUpDate",
    "Flagged completed follow-ups with missing FollowUpDate",
    completed_missing_followup_date_mask.sum(),
    "Exact follow-up date cannot be reliably reconstructed"
)

In [287]:
admissions_clean[
    "DaysToFollowUp"
] = (
    admissions_clean["FollowUpDate"]
    -
    admissions_clean["DischargeDateTime"]
).dt.total_seconds() / 86400

In [288]:
valid_followup_timing_mask = (
    (
        admissions_clean[
            "FollowUpCompletedFlag"
        ] == 1
    )
    &
    (
        admissions_clean[
            "FollowUpDateValidFlag"
        ] == 1
    )
    &
    admissions_clean[
        "FollowUpDate"
    ].notna()
    &
    admissions_clean[
        "DischargeDateTime"
    ].notna()
    &
    (
        admissions_clean[
            "FollowUpDate"
        ]
        >=
        admissions_clean[
            "DischargeDateTime"
        ]
    )
)

In [289]:
admissions_clean[
    "DaysToFollowUp"
] = admissions_clean[
    "DaysToFollowUp"
].where(
    valid_followup_timing_mask,
    np.nan
)

In [290]:
print(
    "Negative DaysToFollowUp:",
    (
        admissions_clean[
            "DaysToFollowUp"
        ] < 0
    ).sum()
)

Negative DaysToFollowUp: 0


In [291]:
readmission_check = (
    admissions_clean[
        admissions_clean[
            "DischargeDisposition"
        ] != "Expired"
    ]
    .sort_values(
        [
            "PatientID",
            "AdmissionDateTime"
        ]
    )
    .copy()
)

readmission_check[
    "NextAdmissionDateTime"
] = (
    readmission_check
    .groupby("PatientID")[
        "AdmissionDateTime"
    ]
    .shift(-1)
)

readmission_check[
    "DaysToNextAdmission"
] = (
    readmission_check[
        "NextAdmissionDateTime"
    ]
    -
    readmission_check[
        "DischargeDateTime"
    ]
).dt.total_seconds() / 86400

In [292]:
temporary_readmission_flag = (
    readmission_check[
        "DaysToNextAdmission"
    ].between(
        0,
        30,
        inclusive="both"
    )
)

print(
    "Temporary 30-day readmission rate:",
    round(
        temporary_readmission_flag.mean()
        * 100,
        2
    ),
    "%"
)

Temporary 30-day readmission rate: 13.45 %


In [293]:
admissions_clean[
    "AdmissionQualityFlag"
] = np.where(
    (
        admissions_clean[
            "EncounterIDValidFlag"
        ].eq(1)
        &
        admissions_clean[
            "LOSValidFlag"
        ].eq(1)
        &
        ~invalid_admission_type_mask
        &
        ~invalid_disposition_mask
    ),
    1,
    0
)

In [294]:
print("=" * 65)
print("ADMISSIONS ETL VALIDATION")
print("=" * 65)

print(
    "Rows:",
    len(admissions_clean)
)

print(
    "Duplicate AdmissionID:",
    admissions_clean[
        "AdmissionID"
    ].duplicated().sum()
)

print(
    "Missing DischargeDateTime:",
    admissions_clean[
        "DischargeDateTime"
    ].isna().sum()
)

print(
    "Discharge before Admission:",
    (
        admissions_clean[
            "DischargeDateTime"
        ]
        <
        admissions_clean[
            "AdmissionDateTime"
        ]
    ).sum()
)

print(
    "Negative LengthOfStayDays:",
    (
        admissions_clean[
            "LengthOfStayDays"
        ] < 0
    ).sum()
)

print(
    "Negative DaysToFollowUp:",
    (
        admissions_clean[
            "DaysToFollowUp"
        ] < 0
    ).sum()
)

ADMISSIONS ETL VALIDATION
Rows: 17740
Duplicate AdmissionID: 0
Missing DischargeDateTime: 0
Discharge before Admission: 0
Negative LengthOfStayDays: 0
Negative DaysToFollowUp: 0


In [295]:
admissions_clean.to_csv(
    CLEANED_DATA_PATH
    / "admissions_clean.csv",
    index=False
)

encounters_clean.to_csv(
    CLEANED_DATA_PATH
    / "encounters_clean.csv",
    index=False
)

In [296]:
cleaning_log_df = pd.DataFrame(
    cleaning_log
)

cleaning_log_df.to_csv(
    CLEANED_DATA_PATH
    / "cleaning_audit_log.csv",
    index=False
)

print(
    "Admissions ETL completed successfully."
)

Admissions ETL completed successfully.


### Clean appointments
### Parse dates

In [297]:
appointments_clean[
    "ScheduledDate"
] = pd.to_datetime(
    appointments_clean[
        "ScheduledDate"
    ],
    format="mixed",
    errors="coerce"
)

appointments_clean[
    "AppointmentDateTime"
] = pd.to_datetime(
    appointments_clean[
        "AppointmentDateTime"
    ],
    format="mixed",
    errors="coerce"
)

In [298]:
print(
    appointments_clean[
        [
            "ScheduledDate",
            "AppointmentDateTime"
        ]
    ].dtypes
)

print(
    "Missing ScheduledDate:",
    appointments_clean[
        "ScheduledDate"
    ].isna().sum()
)

print(
    "Missing AppointmentDateTime:",
    appointments_clean[
        "AppointmentDateTime"
    ].isna().sum()
)

ScheduledDate          datetime64[us]
AppointmentDateTime    datetime64[us]
dtype: object
Missing ScheduledDate: 0
Missing AppointmentDateTime: 0


In [299]:
log_cleaning_action(
    "5.6",
    "appointments",
    "ScheduledDate / AppointmentDateTime",
    "Converted scheduling fields to datetime",
    len(appointments_clean),
    "Required for booking lead-time and scheduling analysis"
)

### Validate primary key

In [300]:
print(
    "Rows:",
    len(appointments_clean)
)

print(
    "Missing AppointmentID:",
    appointments_clean[
        "AppointmentID"
    ].isna().sum()
)

print(
    "Duplicate AppointmentID:",
    appointments_clean[
        "AppointmentID"
    ].duplicated().sum()
)

Rows: 50000
Missing AppointmentID: 0
Duplicate AppointmentID: 0


### Validate PatientID

In [301]:
invalid_appointment_patient_mask = (
    ~appointments_clean[
        "PatientID"
    ].isin(
        patients_clean[
            "PatientID"
        ]
    )
)

In [302]:
print(
    "Appointments with unresolved PatientID:",
    invalid_appointment_patient_mask.sum()
)

Appointments with unresolved PatientID: 107


In [303]:
appointments_clean[
    "PatientIDValidFlag"
] = (
    ~invalid_appointment_patient_mask
).astype(int)

appointments_clean[
    "PatientIDClean"
] = appointments_clean[
    "PatientID"
].where(
    appointments_clean[
        "PatientIDValidFlag"
    ] == 1,
    "UNKNOWN"
)

In [304]:
log_cleaning_action(
    "5.6",
    "appointments",
    "PatientID",
    "Flagged unresolved PatientIDs and mapped them to UNKNOWN for staging",
    invalid_appointment_patient_mask.sum(),
    "Preserve appointment activity while preventing broken patient-dimension relationships"
)

### Validate DepartmentID

In [305]:
invalid_appointment_department_mask = (
    ~appointments_clean[
        "DepartmentID"
    ].isin(
        departments_clean[
            "DepartmentID"
        ]
    )
)

print(
    "Invalid appointment DepartmentID:",
    invalid_appointment_department_mask.sum()
)

Invalid appointment DepartmentID: 0


In [306]:
appointments_clean[
    "DepartmentIDValidFlag"
] = (
    ~invalid_appointment_department_mask
).astype(int)

appointments_clean[
    "DepartmentIDClean"
] = appointments_clean[
    "DepartmentID"
].where(
    appointments_clean[
        "DepartmentIDValidFlag"
    ] == 1,
    "UNKNOWN"
)

### Handle missing ProviderID

In [307]:
missing_appointment_provider_mask = (
    appointments_clean[
        "ProviderID"
    ].isna()
)

print(
    "Missing ProviderID:",
    missing_appointment_provider_mask.sum()
)

Missing ProviderID: 400


In [308]:
invalid_nonnull_provider_mask = (
    appointments_clean[
        "ProviderID"
    ].notna()
    &
    ~appointments_clean[
        "ProviderID"
    ].isin(
        providers_clean[
            "ProviderID"
        ]
    )
)

print(
    "Invalid non-null ProviderID:",
    invalid_nonnull_provider_mask.sum()
)

Invalid non-null ProviderID: 0


In [309]:
appointments_clean[
    "ProviderIDValidFlag"
] = np.where(
    appointments_clean[
        "ProviderID"
    ].isna(),
    0,
    appointments_clean[
        "ProviderID"
    ].isin(
        providers_clean[
            "ProviderID"
        ]
    ).astype(int)
)

In [310]:
appointments_clean[
    "ProviderIDClean"
] = appointments_clean[
    "ProviderID"
].where(
    appointments_clean[
        "ProviderIDValidFlag"
    ] == 1,
    "UNKNOWN"
)

In [311]:
print(
    "Appointments mapped to UNKNOWN Provider:",
    (
        appointments_clean[
            "ProviderIDClean"
        ] == "UNKNOWN"
    ).sum()
)

Appointments mapped to UNKNOWN Provider: 400


In [312]:
log_cleaning_action(
    "5.6",
    "appointments",
    "ProviderID",
    "Mapped missing or invalid provider references to UNKNOWN",
    (
        appointments_clean[
            "ProviderIDClean"
        ] == "UNKNOWN"
    ).sum(),
    "Preserve appointment activity while protecting provider-level reporting relationships"
)

### Validate provider/department consistency

In [313]:
appointment_provider_check = (
    appointments_clean[
        appointments_clean[
            "ProviderIDValidFlag"
        ] == 1
    ]
    .merge(
        providers_clean[
            [
                "ProviderID",
                "DepartmentID"
            ]
        ],
        on="ProviderID",
        how="left",
        suffixes=(
            "_Appointment",
            "_Provider"
        )
    )
)

In [314]:
provider_department_mismatch_mask = (
    appointment_provider_check[
        "DepartmentID_Appointment"
    ]
    !=
    appointment_provider_check[
        "DepartmentID_Provider"
    ]
)

print(
    "Provider / Department mismatches:",
    provider_department_mismatch_mask.sum()
)

Provider / Department mismatches: 0


### Standardize AppointmentStatus

In [315]:
appointments_clean[
    "AppointmentStatus"
].value_counts(
    dropna=False
)

AppointmentStatus
Completed        37025
Cancelled         4981
No Show           4860
Rescheduled       2984
UnknownStatus       47
Canceled            39
NO_SHOW             35
Complete            29
Name: count, dtype: int64

In [316]:
appointment_status_map = {
    "Completed": "Completed",
    "Complete": "Completed",

    "Cancelled": "Cancelled",
    "Canceled": "Cancelled",

    "No Show": "No Show",
    "NO_SHOW": "No Show",

    "Rescheduled": "Rescheduled"
}

In [317]:
appointments_clean[
    "AppointmentStatusSource"
] = appointments_clean[
    "AppointmentStatus"
]

In [318]:
appointments_clean[
    "AppointmentStatusClean"
] = (
    appointments_clean[
        "AppointmentStatus"
    ]
    .map(
        appointment_status_map
    )
)

In [319]:
appointments_clean[
    "AppointmentStatusClean"
] = (
    appointments_clean[
        "AppointmentStatusClean"
    ]
    .fillna("Unknown")
)

In [320]:
appointments_clean[
    [
        "AppointmentStatusSource",
        "AppointmentStatusClean"
    ]
].value_counts()

AppointmentStatusSource  AppointmentStatusClean
Completed                Completed                 37025
Cancelled                Cancelled                  4981
No Show                  No Show                    4860
Rescheduled              Rescheduled                2984
UnknownStatus            Unknown                      47
Canceled                 Cancelled                    39
NO_SHOW                  No Show                      35
Complete                 Completed                    29
Name: count, dtype: int64

### Create status validity flag

In [321]:
valid_appointment_statuses = [
    "Completed",
    "Cancelled",
    "No Show",
    "Rescheduled"
]

appointments_clean[
    "AppointmentStatusValidFlag"
] = (
    appointments_clean[
        "AppointmentStatusClean"
    ].isin(
        valid_appointment_statuses
    )
).astype(int)

In [322]:
unresolved_status_count = (
    appointments_clean[
        "AppointmentStatusClean"
    ] == "Unknown"
).sum()

print(
    "Unresolved AppointmentStatus:",
    unresolved_status_count
)

Unresolved AppointmentStatus: 47


In [323]:
standardized_status_count = (
    appointments_clean[
        "AppointmentStatusSource"
    ]
    !=
    appointments_clean[
        "AppointmentStatusClean"
    ]
).sum()

print(
    "Appointment status values changed:",
    standardized_status_count
)

Appointment status values changed: 150


In [324]:
log_cleaning_action(
    "5.6",
    "appointments",
    "AppointmentStatus",
    "Standardized known status variants and mapped unresolved values to Unknown",
    standardized_status_count,
    "Required for consistent no-show, cancellation, completion, and rescheduling KPIs"
)

### Validate AppointmentType

In [325]:
valid_appointment_types = [
    "New Patient",
    "Follow-Up",
    "Routine",
    "Specialist",
    "Post-Discharge Follow-Up"
]

invalid_appointment_type_mask = (
    ~appointments_clean[
        "AppointmentType"
    ].isin(
        valid_appointment_types
    )
)

print(
    "Invalid AppointmentType:",
    invalid_appointment_type_mask.sum()
)

Invalid AppointmentType: 0


### Detect impossible appointment dates

In [326]:
appointment_before_schedule_mask = (
    appointments_clean[
        "AppointmentDateTime"
    ].notna()
    &
    appointments_clean[
        "ScheduledDate"
    ].notna()
    &
    (
        appointments_clean[
            "AppointmentDateTime"
        ].dt.normalize()
        <
        appointments_clean[
            "ScheduledDate"
        ].dt.normalize()
    )
)

print(
    "Appointment before ScheduledDate:",
    appointment_before_schedule_mask.sum()
)

Appointment before ScheduledDate: 200


### Create BookingLeadTime validity flag

In [327]:
appointments_clean[
    "BookingLeadTimeValidFlag"
] = (
    appointments_clean[
        "AppointmentDateTime"
    ].notna()
    &
    appointments_clean[
        "ScheduledDate"
    ].notna()
    &
    (
        appointments_clean[
            "AppointmentDateTime"
        ].dt.normalize()
        >=
        appointments_clean[
            "ScheduledDate"
        ].dt.normalize()
    )
).astype(int)

### Derive BookingLeadDays

In [328]:
appointments_clean[
    "BookingLeadDays"
] = (
    appointments_clean[
        "AppointmentDateTime"
    ].dt.normalize()
    -
    appointments_clean[
        "ScheduledDate"
    ].dt.normalize()
).dt.days

In [329]:
appointments_clean[
    "BookingLeadDays"
] = appointments_clean[
    "BookingLeadDays"
].where(
    appointments_clean[
        "BookingLeadTimeValidFlag"
    ] == 1,
    np.nan
)

In [330]:
print(
    "Negative BookingLeadDays:",
    (
        appointments_clean[
            "BookingLeadDays"
        ] < 0
    ).sum()
)

print(
    appointments_clean[
        "BookingLeadDays"
    ].describe()
)

Negative BookingLeadDays: 0
count    49800.000000
mean        25.537490
std         16.787894
min          0.000000
25%         11.000000
50%         23.000000
75%         37.000000
max         75.000000
Name: BookingLeadDays, dtype: float64


In [331]:
log_cleaning_action(
    "5.6",
    "appointments",
    "AppointmentDateTime / ScheduledDate",
    "Flagged impossible scheduling chronology and created KPI-safe BookingLeadDays",
    appointment_before_schedule_mask.sum(),
    "Invalid date sequences cannot be reliably reconstructed and must not create negative booking lead time"
)

### Validate cancellation reason logic

In [332]:
cancelled_missing_reason_mask = (
    (
        appointments_clean[
            "AppointmentStatusClean"
        ] == "Cancelled"
    )
    &
    appointments_clean[
        "CancellationReason"
    ].isna()
)

print(
    "Cancelled appointments missing reason:",
    cancelled_missing_reason_mask.sum()
)

Cancelled appointments missing reason: 33


In [333]:
appointments_clean[
    "CancellationReasonValidFlag"
] = 1

appointments_clean.loc[
    cancelled_missing_reason_mask,
    "CancellationReasonValidFlag"
] = 0

In [334]:
appointments_clean[
    "CancellationReasonClean"
] = appointments_clean[
    "CancellationReason"
]

In [335]:
appointments_clean.loc[
    cancelled_missing_reason_mask,
    "CancellationReasonClean"
] = "Unknown / Not Provided"

In [336]:
print(
    appointments_clean[
        appointments_clean[
            "AppointmentStatusClean"
        ] == "Cancelled"
    ][
        "CancellationReasonClean"
    ].value_counts(
        dropna=False
    )
)

CancellationReasonClean
Provider Unavailable      851
Insurance Issue           836
Scheduling Conflict       831
Transportation Issue      829
Other                     820
Patient Request           820
Unknown / Not Provided     33
Name: count, dtype: int64


In [337]:
non_cancelled_with_reason_mask = (
    (
        appointments_clean[
            "AppointmentStatusClean"
        ] != "Cancelled"
    )
    &
    appointments_clean[
        "CancellationReason"
    ].notna()
)

In [338]:
print(
    "Non-cancelled appointments with cancellation reason:",
    non_cancelled_with_reason_mask.sum()
)

Non-cancelled appointments with cancellation reason: 10


In [339]:
appointments_clean.loc[
    non_cancelled_with_reason_mask,
    "CancellationReasonClean"
] = pd.NA

appointments_clean.loc[
    non_cancelled_with_reason_mask,
    "CancellationReasonValidFlag"
] = 0

In [340]:
cancellation_logic_issue_count = (
    cancelled_missing_reason_mask.sum()
    +
    non_cancelled_with_reason_mask.sum()
)

log_cleaning_action(
    "5.6",
    "appointments",
    "AppointmentStatus / CancellationReason",
    "Flagged inconsistent cancellation reason logic and created reporting-safe CancellationReasonClean",
    cancellation_logic_issue_count,
    "Cancellation reasons are only analytically valid when status is Cancelled; unavailable reasons mapped to Unknown / Not Provided"
)

### Clean lab_results

In [341]:
lab_results_clean[
    "ResultDateTime"
] = pd.to_datetime(
    lab_results_clean[
        "ResultDateTime"
    ],
    format="mixed",
    errors="coerce"
)

In [342]:
numeric_lab_columns = [
    "ResultValue",
    "ReferenceLow",
    "ReferenceHigh"
]

for column in numeric_lab_columns:
    lab_results_clean[column] = pd.to_numeric(
        lab_results_clean[column],
        errors="coerce"
    )

In [343]:
print(
    lab_results_clean[
        [
            "ResultDateTime",
            "ResultValue",
            "ReferenceLow",
            "ReferenceHigh"
        ]
    ].dtypes
)

ResultDateTime    datetime64[us]
ResultValue              float64
ReferenceLow             float64
ReferenceHigh            float64
dtype: object


In [344]:
print(
    "Rows:",
    len(lab_results_clean)
)

print(
    "Missing LabResultID:",
    lab_results_clean[
        "LabResultID"
    ].isna().sum()
)

print(
    "Duplicate LabResultID:",
    lab_results_clean[
        "LabResultID"
    ].duplicated().sum()
)

Rows: 100000
Missing LabResultID: 0
Duplicate LabResultID: 0


### Validate EncounterID

In [345]:
invalid_lab_encounter_mask = (
    ~lab_results_clean[
        "EncounterID"
    ].isin(
        encounters_clean[
            "EncounterID"
        ]
    )
)

print(
    "Invalid lab EncounterID:",
    invalid_lab_encounter_mask.sum()
)

Invalid lab EncounterID: 0


In [346]:
lab_results_clean[
    "EncounterIDValidFlag"
] = (
    ~invalid_lab_encounter_mask
).astype(int)

### Validate PatientID

In [347]:
invalid_lab_patient_mask = (
    ~lab_results_clean[
        "PatientID"
    ].isin(
        patients_clean[
            "PatientID"
        ]
    )
)

print(
    "Lab records with unresolved PatientID:",
    invalid_lab_patient_mask.sum()
)

Lab records with unresolved PatientID: 218


In [348]:
lab_results_clean[
    "PatientIDValidFlag"
] = (
    ~invalid_lab_patient_mask
).astype(int)

lab_results_clean[
    "PatientIDClean"
] = lab_results_clean[
    "PatientID"
].where(
    lab_results_clean[
        "PatientIDValidFlag"
    ] == 1,
    "UNKNOWN"
)

### Check lab patient matches encounter patient

In [349]:
lab_patient_check = (
    lab_results_clean
    .merge(
        encounters_clean[
            [
                "EncounterID",
                "PatientID"
            ]
        ],
        on="EncounterID",
        how="left",
        suffixes=(
            "_Lab",
            "_Encounter"
        )
    )
)

In [350]:
lab_patient_mismatch_mask = (
    lab_patient_check[
        "PatientID_Lab"
    ]
    !=
    lab_patient_check[
        "PatientID_Encounter"
    ]
)

print(
    "Lab / Encounter patient mismatches:",
    lab_patient_mismatch_mask.sum()
)

Lab / Encounter patient mismatches: 0


### Define approved lab metadata

In [351]:
lab_reference_map = {
    "HbA1c": {
        "unit": "%",
        "low": 4.0,
        "high": 5.6
    },
    "Glucose": {
        "unit": "mg/dL",
        "low": 70.0,
        "high": 99.0
    },
    "LDL Cholesterol": {
        "unit": "mg/dL",
        "low": 0.0,
        "high": 100.0
    },
    "Creatinine": {
        "unit": "mg/dL",
        "low": 0.6,
        "high": 1.3
    },
    "Hemoglobin": {
        "unit": "g/dL",
        "low": 12.0,
        "high": 17.5
    }
}

In [352]:
lab_results_clean[
    "LabTest"
].value_counts()

LabTest
Glucose            20076
LDL Cholesterol    20058
HbA1c              20042
Creatinine         19957
Hemoglobin         19867
Name: count, dtype: int64

### Recover missing units

In [353]:
lab_results_clean[
    "ResultUnitSource"
] = lab_results_clean[
    "ResultUnit"
]

In [354]:
expected_unit_map = {
    test: config["unit"]
    for test, config
    in lab_reference_map.items()
}

lab_results_clean[
    "ExpectedResultUnit"
] = (
    lab_results_clean[
        "LabTest"
    ].map(
        expected_unit_map
    )
)

In [355]:
missing_unit_mask = (
    lab_results_clean[
        "ResultUnit"
    ].isna()
)

print(
    "Missing ResultUnit before:",
    missing_unit_mask.sum()
)

Missing ResultUnit before: 700


In [356]:
lab_results_clean.loc[
    missing_unit_mask,
    "ResultUnit"
] = (
    lab_results_clean.loc[
        missing_unit_mask,
        "ExpectedResultUnit"
    ]
)

In [357]:
print(
    "Missing ResultUnit after:",
    lab_results_clean[
        "ResultUnit"
    ].isna().sum()
)

Missing ResultUnit after: 0


In [358]:
log_cleaning_action(
    "5.7",
    "lab_results",
    "ResultUnit",
    "Recovered missing units using deterministic LabTest-to-unit mapping",
    missing_unit_mask.sum(),
    "LabTest uniquely determines the expected unit in the synthetic source specification"
)

### Correct invalid units

In [359]:
incorrect_unit_mask = (
    lab_results_clean[
        "ResultUnit"
    ].notna()
    &
    (
        lab_results_clean[
            "ResultUnit"
        ]
        !=
        lab_results_clean[
            "ExpectedResultUnit"
        ]
    )
)

print(
    "Incorrect ResultUnit:",
    incorrect_unit_mask.sum()
)

Incorrect ResultUnit: 300


In [360]:
lab_results_clean.loc[
    incorrect_unit_mask,
    "ResultUnit"
] = lab_results_clean.loc[
    incorrect_unit_mask,
    "ExpectedResultUnit"
]

In [361]:
print(
    "Remaining incorrect ResultUnit:",
    (
        lab_results_clean[
            "ResultUnit"
        ]
        !=
        lab_results_clean[
            "ExpectedResultUnit"
        ]
    ).sum()
)

Remaining incorrect ResultUnit: 0


In [362]:
log_cleaning_action(
    "5.7",
    "lab_results",
    "ResultUnit",
    "Standardized incorrect units using approved LabTest metadata",
    incorrect_unit_mask.sum(),
    "Unit is deterministic from LabTest in the project source specification"
)

### Repair invalid reference ranges

In [363]:
lab_results_clean[
    "ReferenceLowSource"
] = lab_results_clean[
    "ReferenceLow"
]

lab_results_clean[
    "ReferenceHighSource"
] = lab_results_clean[
    "ReferenceHigh"
]

In [364]:
expected_low_map = {
    test: config["low"]
    for test, config
    in lab_reference_map.items()
}

expected_high_map = {
    test: config["high"]
    for test, config
    in lab_reference_map.items()
}

In [365]:
lab_results_clean[
    "ExpectedReferenceLow"
] = (
    lab_results_clean[
        "LabTest"
    ].map(
        expected_low_map
    )
)

lab_results_clean[
    "ExpectedReferenceHigh"
] = (
    lab_results_clean[
        "LabTest"
    ].map(
        expected_high_map
    )
)

In [366]:
invalid_reference_range_mask = (
    lab_results_clean[
        "ReferenceLow"
    ]
    >
    lab_results_clean[
        "ReferenceHigh"
    ]
)

print(
    "Invalid reference ranges:",
    invalid_reference_range_mask.sum()
)

Invalid reference ranges: 200


In [367]:
lab_results_clean.loc[
    invalid_reference_range_mask,
    "ReferenceLow"
] = (
    lab_results_clean.loc[
        invalid_reference_range_mask,
        "ExpectedReferenceLow"
    ]
)

lab_results_clean.loc[
    invalid_reference_range_mask,
    "ReferenceHigh"
] = (
    lab_results_clean.loc[
        invalid_reference_range_mask,
        "ExpectedReferenceHigh"
    ]
)

In [368]:
print(
    "ReferenceLow > ReferenceHigh remaining:",
    (
        lab_results_clean[
            "ReferenceLow"
        ]
        >
        lab_results_clean[
            "ReferenceHigh"
        ]
    ).sum()
)

ReferenceLow > ReferenceHigh remaining: 0


### Detect statistical extreme outliers

In [369]:
lab_results_clean[
    "StatisticalOutlierFlag"
] = 0

In [370]:
lab_outlier_summary = []

for lab_test, group in lab_results_clean.groupby(
    "LabTest"
):

    q1 = group[
        "ResultValue"
    ].quantile(0.25)

    q3 = group[
        "ResultValue"
    ].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 3 * iqr
    upper_bound = q3 + 3 * iqr

    outlier_indices = group[
        (
            group[
                "ResultValue"
            ] < lower_bound
        )
        |
        (
            group[
                "ResultValue"
            ] > upper_bound
        )
    ].index

    lab_results_clean.loc[
        outlier_indices,
        "StatisticalOutlierFlag"
    ] = 1

    lab_outlier_summary.append({
        "LabTest": lab_test,
        "LowerBound": round(
            lower_bound,
            2
        ),
        "UpperBound": round(
            upper_bound,
            2
        ),
        "OutlierCount": len(
            outlier_indices
        )
    })

In [371]:
lab_outlier_summary_df = pd.DataFrame(
    lab_outlier_summary
)

lab_outlier_summary_df

,LabTest,LowerBound,UpperBound,OutlierCount
0,Creatinine,-0.43,2.44,58
1,Glucose,-37.10,248.29,59
2,HbA1c,0.05,12.37,63
3,Hemoglobin,5.67,22.75,59
4,LDL Cholesterol,-49.23,280.10,61


In [372]:
print(
    "Statistical extreme outliers:",
    lab_results_clean[
        "StatisticalOutlierFlag"
    ].sum()
)

Statistical extreme outliers: 300


### Create KPI-safe ResultValueClean

In [373]:
lab_results_clean[
    "ResultValueClean"
] = (
    lab_results_clean[
        "ResultValue"
    ].where(
        lab_results_clean[
            "StatisticalOutlierFlag"
        ] == 0,
        np.nan
    )
)

### Recalculate analytical ResultFlag

In [374]:
lab_results_clean[
    "ResultFlagSource"
] = lab_results_clean[
    "ResultFlag"
]

In [375]:
def calculate_result_flag(
    value,
    low,
    high
):

    if pd.isna(value):
        return "Unknown"

    if value < low:
        return "Low"

    elif value > high * 1.5:
        return "Critical"

    elif value > high:
        return "High"

    else:
        return "Normal"

In [376]:
lab_results_clean[
    "ResultFlagClean"
] = lab_results_clean.apply(
    lambda row: calculate_result_flag(
        row["ResultValue"],
        row["ReferenceLow"],
        row["ReferenceHigh"]
    ),
    axis=1
)

In [377]:
result_flag_changed_mask = (
    lab_results_clean[
        "ResultFlagSource"
    ]
    !=
    lab_results_clean[
        "ResultFlagClean"
    ]
)

print(
    "ResultFlag values recalculated differently:",
    result_flag_changed_mask.sum()
)

ResultFlag values recalculated differently: 275


### Validate ResultDateTime against encounter

In [378]:
lab_date_check = (
    lab_results_clean
    .merge(
        encounters_clean[
            [
                "EncounterID",
                "ArrivalDateTime",
                "EncounterEndDateTime"
            ]
        ],
        on="EncounterID",
        how="left"
    )
)

In [379]:
lab_before_encounter_mask = (
    lab_date_check[
        "ResultDateTime"
    ].notna()
    &
    lab_date_check[
        "ArrivalDateTime"
    ].notna()
    &
    (
        lab_date_check[
            "ResultDateTime"
        ]
        <
        lab_date_check[
            "ArrivalDateTime"
        ]
    )
)

In [380]:
lab_after_encounter_mask = (
    lab_date_check[
        "ResultDateTime"
    ].notna()
    &
    lab_date_check[
        "EncounterEndDateTime"
    ].notna()
    &
    (
        lab_date_check[
            "ResultDateTime"
        ]
        >
        lab_date_check[
            "EncounterEndDateTime"
        ]
    )
)

In [381]:
print(
    "Lab result before encounter:",
    lab_before_encounter_mask.sum()
)

print(
    "Lab result after encounter:",
    lab_after_encounter_mask.sum()
)

Lab result before encounter: 0
Lab result after encounter: 75


In [382]:
lab_after_detail = (
    lab_date_check[
        lab_after_encounter_mask
    ]
    .copy()
)

lab_after_detail[
    "MinutesAfterEncounterEnd"
] = (
    lab_after_detail[
        "ResultDateTime"
    ]
    -
    lab_after_detail[
        "EncounterEndDateTime"
    ]
).dt.total_seconds() / 60

print(
    "Lab results after encounter:",
    len(lab_after_detail)
)

print(
    lab_after_detail[
        "MinutesAfterEncounterEnd"
    ].describe()
)

Lab results after encounter: 75
count      75.000000
mean     2437.412000
std      1648.166852
min        54.000000
25%      1173.700000
50%      2054.000000
75%      3537.100000
max      6303.700000
Name: MinutesAfterEncounterEnd, dtype: float64


In [383]:
lab_after_detail[
    [
        "LabResultID",
        "EncounterID",
        "ResultDateTime",
        "EncounterEndDateTime",
        "MinutesAfterEncounterEnd"
    ]
].head(20)

,LabResultID,EncounterID,ResultDateTime,EncounterEndDateTime,MinutesAfterEncounterEnd
1050,LAB0001051,ENC057634,2025-07-02 09:00:00,2025-06-28 15:59:18,5340.7
2136,LAB0002137,ENC036423,2025-11-12 12:42:00,2025-11-12 04:53:30,468.5
3360,LAB0003361,ENC030990,2024-08-01 00:16:00,2024-07-30 17:34:24,1841.6
9197,LAB0009198,ENC009235,2025-12-05 22:58:00,2025-12-05 11:00:30,717.5
9271,LAB0009272,ENC052438,2024-05-18 01:41:00,2024-05-15 18:31:54,3309.1
10271,LAB0010272,ENC030218,2024-08-04 02:13:00,2024-08-01 12:26:48,3706.2
11778,LAB0011779,ENC027539,2024-03-01 11:05:00,2024-02-26 13:36:24,5608.6
13542,LAB0013543,ENC007428,2025-12-12 18:42:00,2025-12-10 18:44:18,2877.7
14569,LAB0014570,ENC075862,2025-01-28 02:09:00,2025-01-26 23:41:36,1587.4
14597,LAB0014598,ENC070857,2025-10-27 02:16:00,2025-10-26 07:09:12,1146.8


In [384]:
lab_after_detail = (
    lab_after_detail
    .merge(
        encounters_clean[
            [
                "EncounterID",
                "EncounterEndRecoveredFlag"
            ]
        ],
        on="EncounterID",
        how="left"
    )
)

In [385]:
print(
    "After-end labs linked to reconstructed encounters:",
    (
        lab_after_detail[
            "EncounterEndRecoveredFlag"
        ] == 1
    ).sum()
)

print(
    "After-end labs linked to normal encounters:",
    (
        lab_after_detail[
            "EncounterEndRecoveredFlag"
        ] != 1
    ).sum()
)

After-end labs linked to reconstructed encounters: 75
After-end labs linked to normal encounters: 0


In [386]:
print(
    "Within 1 minute:",
    (
        lab_after_detail[
            "MinutesAfterEncounterEnd"
        ] <= 1
    ).sum()
)

print(
    "Within 5 minutes:",
    (
        lab_after_detail[
            "MinutesAfterEncounterEnd"
        ] <= 5
    ).sum()
)

print(
    "Within 60 minutes:",
    (
        lab_after_detail[
            "MinutesAfterEncounterEnd"
        ] <= 60
    ).sum()
)

print(
    "More than 60 minutes:",
    (
        lab_after_detail[
            "MinutesAfterEncounterEnd"
        ] > 60
    ).sum()
)

Within 1 minute: 0
Within 5 minutes: 0
Within 60 minutes: 1
More than 60 minutes: 74


In [388]:
[
    column
    for column in lab_after_with_recovery.columns
    if "EncounterEndRecoveredFlag" in column
]

['EncounterEndRecoveredFlag_x', 'EncounterEndRecoveredFlag_y']

In [389]:
lab_after_with_recovery = (
    lab_after_detail.copy()
)

In [390]:
print(
    "Flag already present:",
    "EncounterEndRecoveredFlag"
    in lab_after_with_recovery.columns
)

Flag already present: True


In [391]:
print(
    "After-end labs linked to reconstructed encounters:",
    (
        lab_after_with_recovery[
            "EncounterEndRecoveredFlag"
        ] == 1
    ).sum()
)

print(
    "After-end labs linked to normal encounters:",
    (
        lab_after_with_recovery[
            "EncounterEndRecoveredFlag"
        ] != 1
    ).sum()
)

After-end labs linked to reconstructed encounters: 75
After-end labs linked to normal encounters: 0


In [393]:
problem_lab_encounter_ids = (
    lab_after_detail[
        "EncounterID"
    ]
    .drop_duplicates()
)

raw_admission_check = (
    admissions_raw[
        admissions_raw[
            "EncounterID"
        ].isin(
            problem_lab_encounter_ids
        )
    ][
        [
            "AdmissionID",
            "EncounterID",
            "AdmissionDateTime",
            "DischargeDateTime"
        ]
    ]
    .copy()
)

print(
    "Unique problem encounters:",
    len(problem_lab_encounter_ids)
)

print(
    "Matching raw admissions:",
    len(raw_admission_check)
)

Unique problem encounters: 49
Matching raw admissions: 49


In [394]:
raw_admission_check[
    "AdmissionDateTimeParsed"
] = pd.to_datetime(
    raw_admission_check[
        "AdmissionDateTime"
    ],
    format="mixed",
    errors="coerce"
)

raw_admission_check[
    "DischargeDateTimeParsed"
] = pd.to_datetime(
    raw_admission_check[
        "DischargeDateTime"
    ],
    format="mixed",
    errors="coerce"
)

In [395]:
raw_admission_check[
    "RawDischargeValidFlag"
] = (
    raw_admission_check[
        "AdmissionDateTimeParsed"
    ].notna()
    &
    raw_admission_check[
        "DischargeDateTimeParsed"
    ].notna()
    &
    (
        raw_admission_check[
            "DischargeDateTimeParsed"
        ]
        >=
        raw_admission_check[
            "AdmissionDateTimeParsed"
        ]
    )
).astype(int)

In [396]:
print(
    "Valid raw discharge timestamps:",
    raw_admission_check[
        "RawDischargeValidFlag"
    ].sum()
)

print(
    "Missing/invalid raw discharge timestamps:",
    (
        raw_admission_check[
            "RawDischargeValidFlag"
        ] == 0
    ).sum()
)

Valid raw discharge timestamps: 0
Missing/invalid raw discharge timestamps: 49


In [397]:
valid_raw_discharge_lookup = (
    raw_admission_check[
        raw_admission_check[
            "RawDischargeValidFlag"
        ] == 1
    ]
    .drop_duplicates(
        subset="EncounterID"
    )
    .set_index(
        "EncounterID"
    )[
        "DischargeDateTimeParsed"
    ]
)

In [399]:
raw_discharge_for_merge = (
    raw_admission_check[
        raw_admission_check[
            "RawDischargeValidFlag"
        ] == 1
    ][
        [
            "EncounterID",
            "DischargeDateTimeParsed"
        ]
    ]
    .drop_duplicates(
        subset="EncounterID"
    )
    .rename(
        columns={
            "DischargeDateTimeParsed":
            "RawAdmissionDischarge"
        }
    )
)

In [400]:
# Remove the column first if it already exists from a failed/rerun cell
lab_after_detail = lab_after_detail.drop(
    columns=["RawAdmissionDischarge"],
    errors="ignore"
)

lab_after_detail = (
    lab_after_detail
    .merge(
        raw_discharge_for_merge,
        on="EncounterID",
        how="left"
    )
)

In [401]:
print(
    "Problem labs with usable raw discharge:",
    lab_after_detail[
        "RawAdmissionDischarge"
    ].notna().sum()
)

print(
    "Labs still after RAW admission discharge:",
    (
        lab_after_detail[
            "RawAdmissionDischarge"
        ].notna()
        &
        (
            lab_after_detail[
                "ResultDateTime"
            ]
            >
            lab_after_detail[
                "RawAdmissionDischarge"
            ]
        )
    ).sum()
)

Problem labs with usable raw discharge: 0
Labs still after RAW admission discharge: 0


In [402]:
print(
    "Unique problem encounters:",
    lab_after_detail[
        "EncounterID"
    ].nunique()
)

print(
    "Valid raw discharge timestamps:",
    raw_admission_check[
        "RawDischargeValidFlag"
    ].sum()
)

Unique problem encounters: 49
Valid raw discharge timestamps: 0


In [403]:
lab_date_check[
    "EncounterEndRecoveredFlag"
] = (
    lab_date_check[
        "EncounterID"
    ]
    .map(
        encounters_clean
        .set_index("EncounterID")[
            "EncounterEndRecoveredFlag"
        ]
    )
    .fillna(0)
    .astype(int)
)

In [404]:
strict_lab_after_encounter_mask = (
    lab_after_encounter_mask
    &
    (
        lab_date_check[
            "EncounterEndRecoveredFlag"
        ] == 0
    )
)

reconstructed_end_timing_exception_mask = (
    lab_after_encounter_mask
    &
    (
        lab_date_check[
            "EncounterEndRecoveredFlag"
        ] == 1
    )
)

In [405]:
print(
    "Strict lab-after-encounter violations:",
    strict_lab_after_encounter_mask.sum()
)

print(
    "Timing exceptions caused by reconstructed encounter end:",
    reconstructed_end_timing_exception_mask.sum()
)

Strict lab-after-encounter violations: 0
Timing exceptions caused by reconstructed encounter end: 75


In [406]:
lab_date_check[
    "ResultDateTimingStatus"
] = "Valid"

In [407]:
lab_date_check.loc[
    lab_before_encounter_mask
    |
    strict_lab_after_encounter_mask,
    "ResultDateTimingStatus"
] = "Invalid"

In [408]:
lab_date_check.loc[
    reconstructed_end_timing_exception_mask,
    "ResultDateTimingStatus"
] = (
    "Valid with Reconstructed End-Time Limitation"
)

In [409]:
lab_date_check[
    "ResultDateTimingStatus"
].value_counts()

ResultDateTimingStatus
Valid                                           99925
Valid with Reconstructed End-Time Limitation       75
Name: count, dtype: int64

In [410]:
lab_timing_status_lookup = (
    lab_date_check
    .set_index("LabResultID")[
        "ResultDateTimingStatus"
    ]
)

In [411]:
lab_results_clean[
    "ResultDateTimingStatus"
] = (
    lab_results_clean[
        "LabResultID"
    ].map(
        lab_timing_status_lookup
    )
)

In [412]:
lab_results_clean[
    "ResultDateValidFlag"
] = (
    lab_results_clean[
        "ResultDateTimingStatus"
    ] != "Invalid"
).astype(int)

In [413]:
lab_results_clean[
    "ResultDateLimitedConfidenceFlag"
] = (
    lab_results_clean[
        "ResultDateTimingStatus"
    ]
    ==
    "Valid with Reconstructed End-Time Limitation"
).astype(int)

In [414]:
print(
    "Invalid lab dates:",
    (
        lab_results_clean[
            "ResultDateValidFlag"
        ] == 0
    ).sum()
)

print(
    "Lab dates with limited timing confidence:",
    lab_results_clean[
        "ResultDateLimitedConfidenceFlag"
    ].sum()
)

Invalid lab dates: 0
Lab dates with limited timing confidence: 75


In [415]:
log_cleaning_action(
    "5.7",
    "lab_results",
    "ResultDateTime",
    "Retained lab results exceeding reconstructed encounter end timestamps with limited-confidence timing flag",
    reconstructed_end_timing_exception_mask.sum(),
    "All upper-bound timing conflicts occurred only on encounters whose end timestamps were reconstructed; no conflicts occurred on encounters with original valid end timestamps"
)

In [416]:
lab_results_clean[
    "LabKPIEligibleFlag"
] = (
    lab_results_clean[
        "EncounterIDValidFlag"
    ].eq(1)
    &
    lab_results_clean[
        "ResultDateValidFlag"
    ].eq(1)
    &
    lab_results_clean[
        "ResultValue"
    ].notna()
).astype(int)

# Post-ETL Validation & Reconciliation

### Rebuild raw and cleaned dictionaries

In [417]:
raw_datasets = {
    "patients": patients_raw,
    "providers": providers_raw,
    "departments": departments_raw,
    "payers": payers_raw,
    "diagnoses": diagnoses_raw,
    "procedures": procedures_raw,
    "encounters": encounters_raw,
    "encounter_diagnoses": encounter_diagnoses_raw,
    "encounter_procedures": encounter_procedures_raw,
    "admissions": admissions_raw,
    "appointments": appointments_raw,
    "lab_results": lab_results_raw
}

cleaned_datasets = {
    "patients": patients_clean,
    "providers": providers_clean,
    "departments": departments_clean,
    "payers": payers_clean,
    "diagnoses": diagnoses_clean,
    "procedures": procedures_clean,
    "encounters": encounters_clean,
    "encounter_diagnoses": encounter_diagnoses_clean,
    "encounter_procedures": encounter_procedures_clean,
    "admissions": admissions_clean,
    "appointments": appointments_clean,
    "lab_results": lab_results_clean
}

### Raw vs cleaned row reconciliation

In [418]:
reconciliation_records = []

for table_name in raw_datasets.keys():

    raw_rows = len(
        raw_datasets[table_name]
    )

    clean_rows = len(
        cleaned_datasets[table_name]
    )

    reconciliation_records.append({
        "Table": table_name,
        "RawRows": raw_rows,
        "CleanedRows": clean_rows,
        "RowDifference": clean_rows - raw_rows,
        "RetentionPercent": round(
            clean_rows / raw_rows * 100,
            2
        ) if raw_rows > 0 else 0
    })

row_reconciliation_df = pd.DataFrame(
    reconciliation_records
)

row_reconciliation_df

,Table,RawRows,CleanedRows,RowDifference,RetentionPercent
0,patients,10000,9980,-20,99.8
1,providers,120,120,0,100.0
2,departments,25,25,0,100.0
3,payers,6,6,0,100.0
4,diagnoses,50,50,0,100.0
5,procedures,40,40,0,100.0
6,encounters,90000,90000,0,100.0
7,encounter_diagnoses,153595,153137,-458,99.7
8,encounter_procedures,70419,70419,0,100.0
9,admissions,17740,17740,0,100.0


### Calculate total record retention

In [419]:
total_raw_rows = sum(
    len(df)
    for df in raw_datasets.values()
)

total_clean_rows = sum(
    len(df)
    for df in cleaned_datasets.values()
)

print(
    "Total raw records:",
    f"{total_raw_rows:,}"
)

print(
    "Total cleaned records:",
    f"{total_clean_rows:,}"
)

print(
    "Overall retained:",
    round(
        total_clean_rows
        / total_raw_rows
        * 100,
        2
    ),
    "%"
)

Total raw records: 491,995
Total cleaned records: 491,517
Overall retained: 99.9 %


### Validate all primary keys again

In [420]:
clean_primary_keys = {
    "patients": "PatientID",
    "providers": "ProviderID",
    "departments": "DepartmentID",
    "payers": "PayerID",
    "diagnoses": "DiagnosisID",
    "procedures": "ProcedureID",
    "encounters": "EncounterID",
    "encounter_diagnoses": "EncounterDiagnosisID",
    "encounter_procedures": "EncounterProcedureID",
    "admissions": "AdmissionID",
    "appointments": "AppointmentID",
    "lab_results": "LabResultID"
}

pk_validation_records = []

for table_name, pk in clean_primary_keys.items():

    df = cleaned_datasets[
        table_name
    ]

    pk_validation_records.append({
        "Table": table_name,
        "PrimaryKey": pk,
        "MissingPK": int(
            df[pk].isna().sum()
        ),
        "DuplicatePK": int(
            df[pk].duplicated().sum()
        )
    })

post_etl_pk_df = pd.DataFrame(
    pk_validation_records
)

post_etl_pk_df

,Table,PrimaryKey,MissingPK,DuplicatePK
0,patients,PatientID,0,0
1,providers,ProviderID,0,0
2,departments,DepartmentID,0,0
3,payers,PayerID,0,0
4,diagnoses,DiagnosisID,0,0
5,procedures,ProcedureID,0,0
6,encounters,EncounterID,0,0
7,encounter_diagnoses,EncounterDiagnosisID,0,0
8,encounter_procedures,EncounterProcedureID,0,0
9,admissions,AdmissionID,0,0


In [421]:
print(
    "Tables with PK problems:",
    (
        (
            post_etl_pk_df["MissingPK"] > 0
        )
        |
        (
            post_etl_pk_df["DuplicatePK"] > 0
        )
    ).sum()
)

Tables with PK problems: 0


### Recheck diagnosis business-key duplicates

In [422]:
remaining_dx_duplicates = (
    encounter_diagnoses_clean[
        encounter_diagnoses_clean[
            "DiagnosisIDValidFlag"
        ] == 1
    ]
    .duplicated(
        subset=[
            "EncounterID",
            "DiagnosisID"
        ]
    )
    .sum()
)

print(
    "Remaining valid Encounter-Diagnosis duplicates:",
    remaining_dx_duplicates
)

Remaining valid Encounter-Diagnosis duplicates: 0


### Validate staging-safe foreign keys

In [423]:
print(
    "Invalid PatientIDClean values:",
    (
        ~encounters_clean[
            "PatientIDClean"
        ].isin(
            list(
                patients_clean[
                    "PatientID"
                ]
            )
            +
            ["UNKNOWN"]
        )
    ).sum()
)

print(
    "Invalid DepartmentIDClean values:",
    (
        ~encounters_clean[
            "DepartmentIDClean"
        ].isin(
            list(
                departments_clean[
                    "DepartmentID"
                ]
            )
            +
            ["UNKNOWN"]
        )
    ).sum()
)

print(
    "Invalid ProviderIDClean values:",
    (
        ~encounters_clean[
            "ProviderIDClean"
        ].isin(
            list(
                providers_clean[
                    "ProviderID"
                ]
            )
            +
            ["UNKNOWN"]
        )
    ).sum()
)

Invalid PatientIDClean values: 0
Invalid DepartmentIDClean values: 0
Invalid ProviderIDClean values: 0


### Diagnosis/procedure staging references

In [424]:
print(
    "Invalid DiagnosisIDClean:",
    (
        ~encounter_diagnoses_clean[
            "DiagnosisIDClean"
        ].isin(
            list(
                diagnoses_clean[
                    "DiagnosisID"
                ]
            )
            +
            ["UNKNOWN"]
        )
    ).sum()
)

Invalid DiagnosisIDClean: 0


In [425]:
print(
    "Invalid ProcedureIDClean:",
    (
        ~encounter_procedures_clean[
            "ProcedureIDClean"
        ].isin(
            list(
                procedures_clean[
                    "ProcedureID"
                ]
            )
            +
            ["UNKNOWN"]
        )
    ).sum()
)

Invalid ProcedureIDClean: 0


### Appointment staging relationships

In [426]:
print(
    "Invalid appointment PatientIDClean:",
    (
        ~appointments_clean[
            "PatientIDClean"
        ].isin(
            list(
                patients_clean[
                    "PatientID"
                ]
            )
            +
            ["UNKNOWN"]
        )
    ).sum()
)

print(
    "Invalid appointment DepartmentIDClean:",
    (
        ~appointments_clean[
            "DepartmentIDClean"
        ].isin(
            list(
                departments_clean[
                    "DepartmentID"
                ]
            )
            +
            ["UNKNOWN"]
        )
    ).sum()
)

print(
    "Invalid appointment ProviderIDClean:",
    (
        ~appointments_clean[
            "ProviderIDClean"
        ].isin(
            list(
                providers_clean[
                    "ProviderID"
                ]
            )
            +
            ["UNKNOWN"]
        )
    ).sum()
)

Invalid appointment PatientIDClean: 0
Invalid appointment DepartmentIDClean: 0
Invalid appointment ProviderIDClean: 0


### Validate KPI-safe numeric fields

In [427]:
print(
    "Negative EncounterCostClean:",
    (
        encounters_clean[
            "EncounterCostClean"
        ] < 0
    ).sum()
)

print(
    "Negative WaitMinutes:",
    (
        encounters_clean[
            "WaitMinutes"
        ] < 0
    ).sum()
)

print(
    "Negative ProcedureCostClean:",
    (
        encounter_procedures_clean[
            "ProcedureCostClean"
        ] < 0
    ).sum()
)

print(
    "Negative LengthOfStayDays:",
    (
        admissions_clean[
            "LengthOfStayDays"
        ] < 0
    ).sum()
)

print(
    "Negative DaysToFollowUp:",
    (
        admissions_clean[
            "DaysToFollowUp"
        ] < 0
    ).sum()
)

print(
    "Negative BookingLeadDays:",
    (
        appointments_clean[
            "BookingLeadDays"
        ] < 0
    ).sum()
)

Negative EncounterCostClean: 0
Negative WaitMinutes: 0
Negative ProcedureCostClean: 0
Negative LengthOfStayDays: 0
Negative DaysToFollowUp: 0
Negative BookingLeadDays: 0


### Validate cleaned lab metadata

In [428]:
print(
    "Missing ResultUnit:",
    lab_results_clean[
        "ResultUnit"
    ].isna().sum()
)

print(
    "Unit mismatch:",
    (
        lab_results_clean[
            "ResultUnit"
        ]
        !=
        lab_results_clean[
            "ExpectedResultUnit"
        ]
    ).sum()
)

print(
    "Invalid reference range:",
    (
        lab_results_clean[
            "ReferenceLow"
        ]
        >
        lab_results_clean[
            "ReferenceHigh"
        ]
    ).sum()
)

print(
    "Strictly invalid ResultDateTime:",
    (
        lab_results_clean[
            "ResultDateValidFlag"
        ] == 0
    ).sum()
)

print(
    "Limited-confidence lab timestamps:",
    lab_results_clean[
        "ResultDateLimitedConfidenceFlag"
    ].sum()
)

Missing ResultUnit: 0
Unit mismatch: 0
Invalid reference range: 0
Strictly invalid ResultDateTime: 0
Limited-confidence lab timestamps: 75


### Show Unknown mappings

In [429]:
unknown_mapping_summary_df = pd.DataFrame([
    {
        "Table": "encounters",
        "Field": "PatientIDClean",
        "UnknownRows": int(
            (
                encounters_clean[
                    "PatientIDClean"
                ] == "UNKNOWN"
            ).sum()
        )
    },
    {
        "Table": "encounters",
        "Field": "DepartmentIDClean",
        "UnknownRows": int(
            (
                encounters_clean[
                    "DepartmentIDClean"
                ] == "UNKNOWN"
            ).sum()
        )
    },
    {
        "Table": "encounters",
        "Field": "ProviderIDClean",
        "UnknownRows": int(
            (
                encounters_clean[
                    "ProviderIDClean"
                ] == "UNKNOWN"
            ).sum()
        )
    },
    {
        "Table": "encounter_diagnoses",
        "Field": "DiagnosisIDClean",
        "UnknownRows": int(
            (
                encounter_diagnoses_clean[
                    "DiagnosisIDClean"
                ] == "UNKNOWN"
            ).sum()
        )
    },
    {
        "Table": "encounter_procedures",
        "Field": "ProcedureIDClean",
        "UnknownRows": int(
            (
                encounter_procedures_clean[
                    "ProcedureIDClean"
                ] == "UNKNOWN"
            ).sum()
        )
    },
    {
        "Table": "appointments",
        "Field": "PatientIDClean",
        "UnknownRows": int(
            (
                appointments_clean[
                    "PatientIDClean"
                ] == "UNKNOWN"
            ).sum()
        )
    },
    {
        "Table": "appointments",
        "Field": "ProviderIDClean",
        "UnknownRows": int(
            (
                appointments_clean[
                    "ProviderIDClean"
                ] == "UNKNOWN"
            ).sum()
        )
    },
    {
        "Table": "lab_results",
        "Field": "PatientIDClean",
        "UnknownRows": int(
            (
                lab_results_clean[
                    "PatientIDClean"
                ] == "UNKNOWN"
            ).sum()
        )
    }
])

unknown_mapping_summary_df

,Table,Field,UnknownRows
0,encounters,PatientIDClean,176
1,encounters,DepartmentIDClean,216
2,encounters,ProviderIDClean,1401
3,encounter_diagnoses,DiagnosisIDClean,459
4,encounter_procedures,ProcedureIDClean,140
5,appointments,PatientIDClean,107
6,appointments,ProviderIDClean,400
7,lab_results,PatientIDClean,218


### Build automated PASS / FAIL report

In [430]:
etl_validation_records = []

def add_validation(
    check_id,
    check_name,
    actual_value,
    expected_value,
    notes
):
    etl_validation_records.append({
        "CheckID": check_id,
        "Check": check_name,
        "Actual": actual_value,
        "Expected": expected_value,
        "Status": (
            "PASS"
            if actual_value == expected_value
            else "REVIEW"
        ),
        "Notes": notes
    })

In [431]:
add_validation(
    "ETL-001",
    "Tables with missing or duplicate PKs",
    int(
        (
            (
                post_etl_pk_df[
                    "MissingPK"
                ] > 0
            )
            |
            (
                post_etl_pk_df[
                    "DuplicatePK"
                ] > 0
            )
        ).sum()
    ),
    0,
    "All cleaned tables require valid unique technical primary keys."
)

add_validation(
    "ETL-002",
    "Valid duplicate Encounter-Diagnosis assignments",
    int(remaining_dx_duplicates),
    0,
    "Confirmed valid duplicate diagnosis assignments should be removed."
)

add_validation(
    "ETL-003",
    "Negative EncounterCostClean",
    int(
        (
            encounters_clean[
                "EncounterCostClean"
            ] < 0
        ).sum()
    ),
    0,
    "KPI-safe encounter cost cannot be negative."
)

add_validation(
    "ETL-004",
    "Negative WaitMinutes",
    int(
        (
            encounters_clean[
                "WaitMinutes"
            ] < 0
        ).sum()
    ),
    0,
    "Wait-time analytical field cannot be negative."
)

add_validation(
    "ETL-005",
    "Negative ProcedureCostClean",
    int(
        (
            encounter_procedures_clean[
                "ProcedureCostClean"
            ] < 0
        ).sum()
    ),
    0,
    "KPI-safe procedure cost cannot be negative."
)

add_validation(
    "ETL-006",
    "Negative LengthOfStayDays",
    int(
        (
            admissions_clean[
                "LengthOfStayDays"
            ] < 0
        ).sum()
    ),
    0,
    "LOS cannot be negative."
)

add_validation(
    "ETL-007",
    "Missing admission discharge timestamps",
    int(
        admissions_clean[
            "DischargeDateTime"
        ].isna().sum()
    ),
    0,
    "Discharge timestamps were reconciled from linked encounter information."
)

add_validation(
    "ETL-008",
    "Negative BookingLeadDays",
    int(
        (
            appointments_clean[
                "BookingLeadDays"
            ] < 0
        ).sum()
    ),
    0,
    "KPI-safe booking lead time cannot be negative."
)

add_validation(
    "ETL-009",
    "Lab unit mismatches",
    int(
        (
            lab_results_clean[
                "ResultUnit"
            ]
            !=
            lab_results_clean[
                "ExpectedResultUnit"
            ]
        ).sum()
    ),
    0,
    "Lab unit should match deterministic project metadata."
)

add_validation(
    "ETL-010",
    "Invalid lab reference ranges",
    int(
        (
            lab_results_clean[
                "ReferenceLow"
            ]
            >
            lab_results_clean[
                "ReferenceHigh"
            ]
        ).sum()
    ),
    0,
    "ReferenceLow cannot exceed ReferenceHigh."
)

In [432]:
etl_validation_report_df = pd.DataFrame(
    etl_validation_records
)

etl_validation_report_df

,CheckID,Check,Actual,Expected,Status,Notes
0,ETL-001,Tables with missing or duplicate PKs,0,0,PASS,All cleaned tables require valid unique techni...
1,ETL-002,Valid duplicate Encounter-Diagnosis assignments,0,0,PASS,Confirmed valid duplicate diagnosis assignment...
2,ETL-003,Negative EncounterCostClean,0,0,PASS,KPI-safe encounter cost cannot be negative.
3,ETL-004,Negative WaitMinutes,0,0,PASS,Wait-time analytical field cannot be negative.
4,ETL-005,Negative ProcedureCostClean,0,0,PASS,KPI-safe procedure cost cannot be negative.
5,ETL-006,Negative LengthOfStayDays,0,0,PASS,LOS cannot be negative.
6,ETL-007,Missing admission discharge timestamps,0,0,PASS,Discharge timestamps were reconciled from link...
7,ETL-008,Negative BookingLeadDays,0,0,PASS,KPI-safe booking lead time cannot be negative.
8,ETL-009,Lab unit mismatches,0,0,PASS,Lab unit should match deterministic project me...
9,ETL-010,Invalid lab reference ranges,0,0,PASS,ReferenceLow cannot exceed ReferenceHigh.


In [433]:
print(
    etl_validation_report_df[
        "Status"
    ].value_counts()
)

Status
PASS    10
Name: count, dtype: int64


### Save the reconciliation artifacts

In [434]:
row_reconciliation_df.to_csv(
    CLEANED_DATA_PATH
    / "etl_row_reconciliation.csv",
    index=False
)

post_etl_pk_df.to_csv(
    CLEANED_DATA_PATH
    / "post_etl_primary_key_validation.csv",
    index=False
)

unknown_mapping_summary_df.to_csv(
    CLEANED_DATA_PATH
    / "unknown_mapping_summary.csv",
    index=False
)

etl_validation_report_df.to_csv(
    CLEANED_DATA_PATH
    / "etl_validation_report.csv",
    index=False
)

In [435]:
cleaning_log_df = pd.DataFrame(
    cleaning_log
)

cleaning_log_df.to_csv(
    CLEANED_DATA_PATH
    / "cleaning_audit_log.csv",
    index=False
)

### Final ETL summary

In [436]:
print("=" * 75)
print("RIVERCARE HEALTH SYSTEM - ETL COMPLETION SUMMARY")
print("=" * 75)

print(
    f"\nSource tables processed: {len(raw_datasets)}"
)

print(
    f"Raw records: {total_raw_rows:,}"
)

print(
    f"Cleaned records: {total_clean_rows:,}"
)

print(
    f"Record retention: "
    f"{round(total_clean_rows / total_raw_rows * 100, 2)}%"
)

print(
    f"Cleaning actions logged: "
    f"{len(cleaning_log_df)}"
)

print(
    f"ETL validation checks passed: "
    f"{(etl_validation_report_df['Status'] == 'PASS').sum()}"
    f"/{len(etl_validation_report_df)}"
)

RIVERCARE HEALTH SYSTEM - ETL COMPLETION SUMMARY

Source tables processed: 12
Raw records: 491,995
Cleaned records: 491,517
Record retention: 99.9%
Cleaning actions logged: 30
ETL validation checks passed: 10/10


In [437]:
appointments_clean.to_csv(
    CLEANED_DATA_PATH / "appointments_clean.csv",
    index=False
)

print(
    "appointments_clean.csv saved successfully."
)

appointments_clean.csv saved successfully.


In [438]:
print(
    (CLEANED_DATA_PATH / "appointments_clean.csv").exists()
)

True


In [439]:
lab_results_clean.to_csv(
    CLEANED_DATA_PATH / "lab_results_clean.csv",
    index=False
)

print(
    "lab_results_clean.csv saved successfully."
)

lab_results_clean.csv saved successfully.


In [469]:
# ------------------------------------------------------------
# Correct stale admission discharge timestamps
# ------------------------------------------------------------

stale_discharge_mask = (
    admissions_clean["LengthOfStayDays"] > 10
)

# Map corrected encounter end timestamps
corrected_discharge_lookup = (
    encounters_clean
    .set_index("EncounterID")[
        "EncounterEndDateTime"
    ]
)

corrected_discharge_times = (
    admissions_clean["EncounterID"]
    .map(corrected_discharge_lookup)
)

# Only update confirmed extreme records where the corrected
# encounter end produces a reasonable LOS <= 10 days
corrected_los_days = (
    corrected_discharge_times
    -
    admissions_clean["AdmissionDateTime"]
).dt.total_seconds() / 86400

stale_discharge_mask = (
    stale_discharge_mask
    &
    corrected_discharge_times.notna()
    &
    (corrected_los_days >= 0)
    &
    (corrected_los_days <= 10)
)

print(
    "Confirmed stale discharge timestamps to correct:",
    stale_discharge_mask.sum()
)

admissions_clean.loc[
    stale_discharge_mask,
    "DischargeDateTime"
] = corrected_discharge_times.loc[
    stale_discharge_mask
]

# Recalculate LOS after correction
admissions_clean["LengthOfStayDays"] = (
    admissions_clean["DischargeDateTime"]
    -
    admissions_clean["AdmissionDateTime"]
).dt.total_seconds() / 86400

Confirmed stale discharge timestamps to correct: 23


In [470]:
print(
    "LOS > 10 days after correction:",
    (
        admissions_clean["LengthOfStayDays"] > 10
    ).sum()
)

print(
    "Negative LOS:",
    (
        admissions_clean["LengthOfStayDays"] < 0
    ).sum()
)

print(
    "Maximum LOS after correction:",
    admissions_clean["LengthOfStayDays"].max()
)

print("\nLOS percentiles:")

print(
    admissions_clean["LengthOfStayDays"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
    .round(2)
)

LOS > 10 days after correction: 0
Negative LOS: 0
Maximum LOS after correction: 7.121889936550926

LOS percentiles:
0.50    3.98
0.75    5.49
0.90    6.41
0.95    6.73
0.99    6.97
1.00    7.12
Name: LengthOfStayDays, dtype: float64


In [471]:
admissions_clean.to_csv(
    CLEANED_DATA_PATH
    / "admissions_clean.csv",
    index=False
)

encounters_clean.to_csv(
    CLEANED_DATA_PATH
    / "encounters_clean.csv",
    index=False
)

In [472]:
saved_admissions = pd.read_csv(
    CLEANED_DATA_PATH / "admissions_clean.csv"
)

saved_encounters = pd.read_csv(
    CLEANED_DATA_PATH / "encounters_clean.csv"
)

print("Saved admissions rows:", len(saved_admissions))
print("Saved encounters rows:", len(saved_encounters))

print(
    "Saved admissions LOS > 10 days:",
    (
        pd.to_numeric(
            saved_admissions["LengthOfStayDays"],
            errors="coerce"
        ) > 10
    ).sum()
)

print(
    "Saved negative LOS:",
    (
        pd.to_numeric(
            saved_admissions["LengthOfStayDays"],
            errors="coerce"
        ) < 0
    ).sum()
)

print(
    "Saved maximum LOS:",
    pd.to_numeric(
        saved_admissions["LengthOfStayDays"],
        errors="coerce"
    ).max()
)

print(
    "Saved missing encounter ends:",
    saved_encounters["EncounterEndDateTime"]
    .isna()
    .sum()
)

Saved admissions rows: 17740
Saved encounters rows: 90000
Saved admissions LOS > 10 days: 0
Saved negative LOS: 0
Saved maximum LOS: 7.121889936550926
Saved missing encounter ends: 0
